# Deep Learning project

Team members:

* Rimsha Afzal
* Nika Sharifi Dariani
* Irina Krylova

## Setup

In [ ]:
from pathlib import Path
from typing import Any, Iterable

import csv
import hashlib
import itertools
import json
import sys
import zipfile

import gdown
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm
from torchvision.datasets import CelebA
from transformers import CLIPModel, CLIPProcessor

In [ ]:
# ============================================================
# Global configuration
# ============================================================

PROJECT_ROOT = Path("/content")

DATA_FILE_ID = "1MP5k9kOUugINr3h7Cc8vNEAikOlBQT91"
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 256
top_k = (1, 5, 10)

GRID_VALUES = [0.25, 0.5, 1.0, 1.5, 2.0]

COMPUTE_EMBEDDINGS = False

save_predictions = False
save_run_config = False

DATA_ROOT = PROJECT_ROOT / "data"
CELEBA_ROOT = DATA_ROOT / "celeba_full"
EMBEDDING_ROOT = CELEBA_ROOT / "embeddings"
QUERY_ROOT = CELEBA_ROOT / "queries"

baseline_test_query_json = QUERY_ROOT / "celeba_evaluation.json"
baseline_val_query_json = QUERY_ROOT / "celeba_validation_evaluation.json"

baseline_img_embedding_train_dir = EMBEDDING_ROOT / "train"
baseline_img_embedding_val_dir = EMBEDDING_ROOT / "valid"
baseline_img_embedding_test_dir = EMBEDDING_ROOT / "test"

baseline_txt_embedding_dir = EMBEDDING_ROOT
visual_direction_dir = EMBEDDING_ROOT
hybrid_text_embedding_dir = EMBEDDING_ROOT
hybrid_visual_direction_dir = EMBEDDING_ROOT
prompt_embedding_dir = EMBEDDING_ROOT
reliability_embedding_dir = EMBEDDING_ROOT
reliability_train_embedding_dir = EMBEDDING_ROOT / "train"

direction_decorrelation_dir = EMBEDDING_ROOT / "decorrelated_directions_thr0p70_max2"

TMP_OUTPUT_ROOT = PROJECT_ROOT / "tmp_outputs"


We upload precomputed embeddings in a zipped folder on Google Drive.

The structure of the full CelebA dataset folders in "data" folder is the following:

```
data/
|--celeba_full/
|    |--artifacts/
|    |--embeddings/
|    |    |--test/
|    |    |    |--image_embeddings.pt
|    |    |    |--image_embeddings_ids.npy
|    |    |--train/
|    |    |    |--image_embeddings.pt
|    |    |    |--image_embeddings_ids.npy
|    |    |--valid/
|    |    |    |--image_embeddings.pt
|    |    |    |--image_embeddings_ids.npy
|    |    |--text_embeddings.pt
|    |    |--text_embeddings_ids.npy
|    |--queries/
|    |    |--celeba_evaluation.json
|    |    |--celeba_validation_evaluation.json
|    |--splits/
```

In [ ]:
# data.zip on Google Drive, shared as "anyone with the link"
if not DATA_ROOT.exists():
    !gdown {DATA_FILE_ID} -O /content/data.zip
    !unzip -q /content/data.zip -d /content

assert (baseline_img_embedding_val_dir / "image_embeddings.pt").exists()


Downloading...
From (original): https://drive.google.com/uc?id=1MP5k9kOUugINr3h7Cc8vNEAikOlBQT91
From (redirected): https://drive.google.com/uc?id=1MP5k9kOUugINr3h7Cc8vNEAikOlBQT91&confirm=t&uuid=430ea50f-fc7c-4fe9-b5ef-7f14a5d3f4a9
To: /content/data.zip
100% 695M/695M [00:08<00:00, 85.6MB/s]


### Embeddings, Normalization and Evaluation

Small shared helpers for loading saved embeddings and normalizing vectors.

In [ ]:
def l2_normalize(embeddings: torch.Tensor) -> torch.Tensor:
    """Scale one embedding or a batch of embeddings to unit length."""
    single = embeddings.dim() == 1
    if single:
        embeddings = embeddings.unsqueeze(0)
    normalized = F.normalize(embeddings.float(), p=2, dim=1, eps=1e-12)
    return normalized.squeeze(0) if single else normalized


def load_embeddings(output_dir: str | Path, name: str, device: str = "cpu") -> tuple[torch.Tensor, list[Any]]:
    """Load a saved tensor and its matching row ids from disk."""
    output_dir = Path(output_dir)
    embeddings_path = output_dir / f"{name}.pt"
    ids_path = output_dir / f"{name}_ids.npy"

    if not embeddings_path.exists():
        raise FileNotFoundError(f"Missing embeddings file: {embeddings_path}")
    if not ids_path.exists():
        raise FileNotFoundError(f"Missing embedding ids file: {ids_path}")

    try:
        embeddings = torch.load(embeddings_path, map_location=device, weights_only=True)
    except TypeError:
        embeddings = torch.load(embeddings_path, map_location=device)
    ids = np.load(ids_path, allow_pickle=True).tolist()
    return embeddings, ids

In [ ]:
# evaluation helper functions
def recall_at_k(
    retrieved_indices: list[int],
    ground_truth_indices: set[int],
    k: int,
) -> float:
    """Return 1.0 if a valid target appears in the top K results."""
    if k <= 0:
        raise ValueError("k must be positive.")

    top_k = set(retrieved_indices[:k])
    return 1.0 if top_k & ground_truth_indices else 0.0


def precision_at_k(
    retrieved_indices: list[int],
    ground_truth_indices: set[int],
    k: int,
) -> float:
    """Return the fraction of top K results that are valid targets."""
    if k <= 0:
        raise ValueError("k must be positive.")

    top_k = retrieved_indices[:k]
    if len(top_k) == 0:
        return 0.0

    hits = sum(1 for idx in top_k if idx in ground_truth_indices)
    return hits / k


def evaluate_single_ranking(
    retrieved_indices: list[int],
    ground_truth_indices: Iterable[int],
    ks: tuple[int, ...] = (1, 5, 10),
) -> dict[str, float]:
    """Compute Recall@K and Precision@K for one source image."""
    ground_truth_set = set(int(idx) for idx in ground_truth_indices)
    metrics = {}

    for k in ks:
        metrics[f"recall@{k}"] = recall_at_k(
            retrieved_indices=retrieved_indices,
            ground_truth_indices=ground_truth_set,
            k=k,
        )
        metrics[f"precision@{k}"] = precision_at_k(
            retrieved_indices=retrieved_indices,
            ground_truth_indices=ground_truth_set,
            k=k,
        )

    return metrics

def average_metrics(metric_rows: list[dict[str, float]]) -> dict[str, float]:
    """Average metric dictionaries across source images."""
    if len(metric_rows) == 0:
        raise ValueError("Cannot average an empty list of metric rows.")

    averaged = {}
    for name in metric_rows[0].keys():
        averaged[name] = sum(row[name] for row in metric_rows) / len(metric_rows)

    return averaged

### Recomputing the embeddings from raw CelebA (optional)

All experiments below use the precomputed CLIP embeddings from `data.zip`. Setting `COMPUTE_EMBEDDINGS = True` in the next cell reproduces them from scratch: it downloads the raw CelebA images from a personal Drive (where they were uploaded manually from the official Drive link) and re-encodes every split with CLIP ViT-B/32 (`openai/clip-vit-base-patch32`). The recomputed tensors are written to the same locations the zip provides (`data/celeba_full/embeddings/...`), so every later section transparently uses them with no changes.

Before overwriting, each recomputed tensor is verified against the stored one: ids and shapes must match exactly, and every row must have cosine similarity $\approx 1$ with its stored counterpart.


In [ ]:
def load_celeba_split(data_root: str | Path, split: str = "test", target_type: str = "attr") -> CelebA:
    """Load a torchvision CelebA split without downloading data."""
    return CelebA(root=str(Path(data_root)), split=split, target_type=target_type, download=False)

raw_data_root = DATA_ROOT / "raw"  # torchvision layout: data/raw/celeba/...

if COMPUTE_EMBEDDINGS:
    celeba_dir = raw_data_root / "celeba"
    celeba_dir.mkdir(parents=True, exist_ok=True)

    # Personal Drive re-upload of the official img_align_celeba.zip
    # The md5 check below proves the copy
    # is byte-identical to the official archive, as published in torchvision.
    IMAGES_FILE_ID = "1tTZjSo4PtECM9tCccwcPBUCua_gJSS_o" # img_align_celeba.zip downloaded manually and uploaded on drive
    IMAGES_MD5 = "00d2c5bc6d35e252742224ab0c1e8fcb"

    if not (celeba_dir / "img_align_celeba").is_dir():
        zip_path = celeba_dir / "img_align_celeba.zip"
        if not zip_path.exists():
            gdown.download(id=IMAGES_FILE_ID, output=str(zip_path), quiet=False)
        digest = hashlib.md5()
        with zip_path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1 << 20), b""):
                digest.update(chunk)
        assert digest.hexdigest() == IMAGES_MD5, f"unexpected md5 for img_align_celeba.zip: {digest.hexdigest()}"
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(celeba_dir)

    # Annotation files from the official CelebA Drive folder (same ids torchvision
    # uses). The CelebA constructor below verifies each file's md5 itself.
    ANNOTATION_FILES = [
        ("0B7EVK8r0v71pblRyaVFSWGxPY0U", "list_attr_celeba.txt"),
        ("0B7EVK8r0v71pY0NSMzRuSXJEVkk", "list_eval_partition.txt"),
        ("1_ee_0u7vcNLOfNLegJRHmolfH5ICW-XS", "identity_CelebA.txt"),
        ("0B7EVK8r0v71pbThiMVRxWXZ4dU0", "list_bbox_celeba.txt"),
        ("0B7EVK8r0v71pd0FJY3Blby1HUTQ", "list_landmarks_align_celeba.txt"),
    ]
    for file_id, filename in ANNOTATION_FILES:
        if not (celeba_dir / filename).exists():
            gdown.download(id=file_id, output=str(celeba_dir / filename), quiet=False)

    split_sizes = {split: len(load_celeba_split(raw_data_root, split)) for split in ("train", "valid", "test")}
    print(split_sizes)
    assert split_sizes == {"train": 162770, "valid": 19867, "test": 19962}


Downloading...
From (original): https://drive.google.com/uc?id=1tTZjSo4PtECM9tCccwcPBUCua_gJSS_o
From (redirected): https://drive.google.com/uc?id=1tTZjSo4PtECM9tCccwcPBUCua_gJSS_o&confirm=t&uuid=e37d229a-0f38-4cab-ab3e-1073cce12c6b
To: /content/data/raw/celeba/img_align_celeba.zip
100%|██████████| 1.44G/1.44G [00:25<00:00, 56.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=0B7EVK8r0v71pblRyaVFSWGxPY0U
To: /content/data/raw/celeba/list_attr_celeba.txt
100%|██████████| 26.7M/26.7M [00:00<00:00, 63.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=0B7EVK8r0v71pY0NSMzRuSXJEVkk
To: /content/data/raw/celeba/list_eval_partition.txt
100%|██████████| 2.84M/2.84M [00:00<00:00, 18.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1_ee_0u7vcNLOfNLegJRHmolfH5ICW-XS
To: /content/data/raw/celeba/identity_CelebA.txt
100%|██████████| 3.42M/3.42M [00:00<00:00, 21.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=0B7EVK8r0v71pbThiMVRxWXZ4dU0
To: /content/data/raw/ce

{'train': 162770, 'valid': 19867, 'test': 19962}


In [ ]:
# CLIP embedding extraction; only used when COMPUTE_EMBEDDINGS is True.

def save_embeddings(embeddings: torch.Tensor, ids: list[Any], output_dir: str | Path, name: str) -> None:
    """Save embeddings as a .pt tensor and matching row ids as a .npy file."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    torch.save(embeddings.cpu().float(), output_dir / f"{name}.pt")
    np.save(output_dir / f"{name}_ids.npy", np.array(ids))


CELEBA_ATTRIBUTES = [
    "5_o_Clock_Shadow", "Arched_Eyebrows", "Attractive",
    "Bags_Under_Eyes", "Bald", "Bangs", "Big_Lips", "Big_Nose",
    "Black_Hair", "Blond_Hair", "Blurry", "Brown_Hair",
    "Bushy_Eyebrows", "Chubby", "Double_Chin", "Eyeglasses",
    "Goatee", "Gray_Hair", "Heavy_Makeup", "High_Cheekbones",
    "Male", "Mouth_Slightly_Open", "Mustache", "Narrow_Eyes",
    "No_Beard", "Oval_Face", "Pale_Skin", "Pointy_Nose",
    "Receding_Hairline", "Rosy_Cheeks", "Sideburns", "Smiling",
    "Straight_Hair", "Wavy_Hair", "Wearing_Earrings", "Wearing_Hat",
    "Wearing_Lipstick", "Wearing_Necklace", "Wearing_Necktie", "Young",
]


def attribute_to_prompt(attribute: str) -> str:
    """Convert a CelebA attribute name into a CLIP text prompt."""
    return f"a photo of a person who is {attribute.replace('_', ' ').lower()}"


def load_clip(model_name: str, device: str):
    """Load the CLIP model and processor on the requested device."""

    resolved_device = torch.device(device)
    processor = CLIPProcessor.from_pretrained(model_name)
    model = CLIPModel.from_pretrained(model_name).to(resolved_device)
    model.eval()
    return model, processor, resolved_device


def _as_tensor(features) -> torch.Tensor:
    """Return the tensor inside a CLIP output object, if needed."""
    return getattr(features, "pooler_output", features)


def extract_image_embeddings(
    data_root: str | Path,
    split: str,
    model_name: str,
    output_dir: str | Path,
    batch_size: int = 64,
    device: str = "cpu",
    max_images: int | None = None,
) -> tuple[torch.Tensor, list[int]]:
    """Encode one CelebA split as normalized CLIP image embeddings."""
    model, processor, resolved_device = load_clip(model_name, device)
    dataset = load_celeba_split(data_root=data_root, split=split, target_type="attr")
    limit = len(dataset) if max_images is None else min(max_images, len(dataset))

    embeddings = []
    ids = []
    for start in tqdm(range(0, limit, batch_size), desc=f"Encoding {split} images"):
        batch_indices = list(range(start, min(start + batch_size, limit)))
        images = [dataset[index][0].convert("RGB") for index in batch_indices]
        inputs = processor(images=images, return_tensors="pt", padding=True).to(resolved_device)
        with torch.no_grad():
            features = _as_tensor(model.get_image_features(**inputs))
        embeddings.append(features.cpu())
        ids.extend(batch_indices)

    image_embeddings = l2_normalize(torch.cat(embeddings, dim=0))
    save_embeddings(image_embeddings, ids, Path(output_dir) / split, "image_embeddings")
    return image_embeddings, ids


def extract_text_embeddings(
    model_name: str,
    output_dir: str | Path,
    batch_size: int = 64,
    device: str = "cpu",
) -> tuple[torch.Tensor, list[str]]:
    """Encode all CelebA attributes as normalized CLIP text embeddings."""
    model, processor, resolved_device = load_clip(model_name, device)
    prompts = [attribute_to_prompt(attribute) for attribute in CELEBA_ATTRIBUTES]

    embeddings = []
    for start in tqdm(range(0, len(prompts), batch_size), desc="Encoding text prompts"):
        batch = prompts[start:start + batch_size]
        inputs = processor(
            text=batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77,
        ).to(resolved_device)
        with torch.no_grad():
            features = _as_tensor(model.get_text_features(**inputs))
        embeddings.append(features.cpu())

    text_embeddings = l2_normalize(torch.cat(embeddings, dim=0))
    save_embeddings(text_embeddings, prompts, output_dir, "text_embeddings")
    return text_embeddings, prompts


In [ ]:
if COMPUTE_EMBEDDINGS:
    embedding_root = EMBEDDING_ROOT
    clip_model_name = CLIP_MODEL_NAME
    extraction_device = device
    print("extraction device:", extraction_device)

    def verify_against_stored(stored, stored_ids, computed, computed_ids, name):
        """Recomputed embeddings must match the stored ones up to float noise."""
        assert list(stored_ids) == list(computed_ids), f"{name}: id mismatch"
        assert stored.shape == computed.shape, f"{name}: shape mismatch"
        cosine = F.cosine_similarity(stored.float(), computed.float())
        print(f"{name}: cosine vs stored min {cosine.min():.4f}, mean {cosine.mean():.4f}")
        assert cosine.min() > 0.99, f"{name}: recomputed embeddings disagree with the stored ones"

    # The stored copies are loaded into memory first because extraction
    # overwrites them on disk; verification then compares against the loaded copies.
    stored_text, stored_text_ids = load_embeddings(embedding_root, "text_embeddings", "cpu")
    text_embeddings, text_ids = extract_text_embeddings(
        model_name=clip_model_name, output_dir=embedding_root, device=extraction_device
    )
    verify_against_stored(stored_text, stored_text_ids, text_embeddings, text_ids, "text_embeddings")

    for split in ("train", "valid", "test"):
        stored_images, stored_image_ids = load_embeddings(embedding_root / split, "image_embeddings", "cpu")
        image_embeddings, image_ids = extract_image_embeddings(
            data_root=raw_data_root,
            split=split,
            model_name=clip_model_name,
            output_dir=embedding_root,
            batch_size=64,
            device=extraction_device,
        )
        verify_against_stored(stored_images, stored_image_ids, image_embeddings, image_ids, f"image_embeddings/{split}")


extraction device: cuda


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

Encoding valid images:   0%|          | 0/311 [00:00<?, ?it/s]

image_embeddings/valid: cosine vs stored min 1.0000, mean 1.0000


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Encoding test images:   0%|          | 0/312 [00:00<?, ?it/s]

image_embeddings/test: cosine vs stored min 1.0000, mean 1.0000


# 1. Introduction

Compositional image retrieval addresses the problem of retrieving target images that preserve a reference image while satisfying requested semantic modifications. In this project, we study this task on CelebA, where each query consists of a source face image together with positive and negative textual attribute conditions such as `+Smiling` or `-Eyeglasses`.

The goal is to rank candidate target images from the official CelebA test split so that the retrieved results both remain visually related to the reference image and reflect the requested attribute changes. We build on the required CLIP ViT-B/32 model (Radford et al., 2021) and evaluate retrieval using the provided ground-truth query file with Recall@K and Precision@K metrics.

Starting from a training-free CLIP text-arithmetic baseline, our methodology progressively studies gated condition weighting, visual attribute directions estimated from CelebA image embeddings, and a hybrid text-and-visual fusion strategy. This allows us to investigate whether combining CLIP's language-based semantic edits with dataset-specific visual evidence can improve compositional retrieval without training a new neural network.

# 2. Method

We employ a training-free approach based on gated fusion. Our modifications study how the reference image, textual constraints, and visual attribute directions should be weighted when constructing the retrieval query.

We used the official CelebA data splits with the following roles:
| Split | Images | Role in our pipeline |
|---|---|---|
| Train | 162,770 | Estimating visual attribute directions (per-attribute mean differences) |
| Validation | 19,867 | Hyperparameter selection (grid search over gate weights) |
| Test | 19,962 | Final evaluation only, against the official ground truth; retrieval gallery for test runs |
| **Total** | **202,599** | |


## 2.1. Baseline

The baseline was implemented as a training-free CLIP text-arithmetic retrieval method using the required `openai/clip-vit-base-patch32` model (Radford et al., 2021).

For each official query, we parse the signed attributes into positive attributes $\mathcal{P}$ and negative attributes $\mathcal{N}$, then construct the composed query vector as

$$
\mathbf{q} = \frac{\tilde{\mathbf{q}}}{\lVert \tilde{\mathbf{q}} \rVert_2},
\qquad
\tilde{\mathbf{q}} = \mathbf{v}_{\text{ref}}
+ \alpha \sum_{p \in \mathcal{P}} \mathbf{t}_p
- \beta \sum_{n \in \mathcal{N}} \mathbf{t}_n,
\qquad \alpha = \beta = 1,
$$

where $\mathbf{v}_{\text{ref}} \in \mathbb{R}^{512}$ is the CLIP image embedding of the reference image; $\mathcal{P}$ and $\mathcal{N}$ are the sets of positive and negative attributes in the query (e.g. $+\textit{Smiling} \in \mathcal{P}$, $-\textit{Eyeglasses} \in \mathcal{N}$); and $\mathbf{t}_p, \mathbf{t}_n$ are the corresponding CLIP text embeddings which are computed using a parsed prompt template "a photo of a person {condition}". All embeddings are L2-normalized, so retrieval ranks the gallery by cosine similarity, computed as a dot product. The reference image itself was excluded from retrieval, and the ranked list was evaluated against the provided CelebA ground-truth targets. This follows the CLAY-style idea that semantic edits can be approximated by arithmetic in CLIP space.

Thus, the baseline treats reference image, negative and positive conditions with the same importance. However, this might be suboptimal as the reference image vector might require different pushes in negative and positive directions. This is why we implement gates to control the weights of conditions.

## 2.2. Gated fusion grid search

### 2.2.1. Grid search strategy

For parameter tuning in every step of our methodology, we run a simple grid search to find optimal values of the gates. The grid search uses the grid of (0.25, 0.5, 1, 1.5, 2) for all of the parameters throughout and is run on the validation split. Then, the best combination is evaluated on the official CelebA test split. This procedure of parameter tuning is also repeated for the next modifications.

The ground truth for the test split is taken from the provided JSON for the assignment. The validation split ground truth was generated by us following the procedure in the assignment. The following are the characteristics of evaluation protocols:

| Split | Query types | Query instances | Valid source–target pairs | Gallery size |
|---|---|---|---|---|
| Validation | 14 | 35,973 | 1,005,074 | 19,867 |
| Test | 14 | 33,052 | 851,111 | 19,962 |

### 2.2.2 Gated fusion

In this first implementation, we run a grid search to tune weights $\alpha$ and $\beta$, corresponding to the weights of positive and negative conditions respectively. The weight of the reference image is kept equal to 1.

The gated variant generalizes the baseline by re-weighting the contributions, keeping the same form but treating $\alpha$ (positive conditions) and $\beta$ (negative conditions) as tunable gates rather than fixing them to $1$:

$$
\mathbf{q} = \frac{\tilde{\mathbf{q}}}{\lVert \tilde{\mathbf{q}} \rVert_2},
\qquad
\tilde{\mathbf{q}} = \gamma\,\mathbf{v}_{\text{ref}}
+ \alpha \sum_{p \in \mathcal{P}} \mathbf{t}_p
- \beta \sum_{n \in \mathcal{N}} \mathbf{t}_n.
$$

Since the final L2-normalization makes the query invariant to a global rescaling of $\tilde{\mathbf{q}}$, only the ratios of the gates are identifiable; we therefore fix the reference-image gate $\gamma = 1$ and select $(\alpha, \beta)$ by grid search, retaining the pair that maximizes Recall@$10$ on the validation split. The baseline is recovered as the special case $\gamma = \alpha = \beta = 1$.

However, this modification does not address one of the most important issues: the modality gap of CLIP embedding space, as we still compute the condition embeddings from text.

## 2.3. Visual directions

The modality gap is an issue because text and visual direction are located in separate narrow "cones" in the embedding space, which is optimal for CLIP's contrastive loss training but can reduce performance (Liang et al. 2022). Another side of this phenomenon is the fact that both visual and text embeddings have a narrow width of the their spanned spaces. Thus, adding text embeddings to an image embedding presumably moves the query vector in the gap direction instead of producing a semantically meaningful change.

However, if we substitute text-based direction with visual directions, then both the reference and the conditions embeddings will be located in the same cone, allowing for a meaningful change.

Specifically, we replace the CLIP text embeddings with *visual* attribute directions. For each CelebA attribute $a$, the direction is estimated on the train split as the difference between the mean image embedding of positive and negative examples:

$$
\mathbf{d}_a = \frac{\boldsymbol{\delta}_a}{\lVert \boldsymbol{\delta}_a \rVert_2},
\qquad
\boldsymbol{\delta}_a =
\frac{1}{|\mathcal{I}_a^{+}|}\sum_{i \in \mathcal{I}_a^{+}} \mathbf{v}_i
-
\frac{1}{|\mathcal{I}_a^{-}|}\sum_{i \in \mathcal{I}_a^{-}} \mathbf{v}_i,
$$

where $\mathcal{I}_a^{+}$ and $\mathcal{I}_a^{-}$ index the train images with attribute $a$ present ($+1$) and absent ($-1$), and $\mathbf{v}_i$ is the CLIP image embedding of image $i$. The query is then formed from the reference image and the signed visual directions, with $\alpha$ gating the positive directions and $\beta$ the negative ones:

$$
\mathbf{q} = \frac{\tilde{\mathbf{q}}}{\lVert \tilde{\mathbf{q}} \rVert_2},
\qquad
\tilde{\mathbf{q}} = \gamma\,\mathbf{v}_{\text{ref}}
+ \alpha \sum_{p \in \mathcal{P}} \mathbf{d}_p
- \beta \sum_{n \in \mathcal{N}} \mathbf{d}_n.
$$

Note that each single-attribute direction $\mathbf{d}_a$ is *itself* a difference of positive and negative image means, so the per-attribute subtraction is baked into $\mathbf{d}_a$ and is independent of the $+/-$ sign carried by the query.

## 2.4. Combined text and visual fusion



### 2.4.1. Motivation and formulation

The combined fusion method extends the CLIP text-arithmetic baseline by adding dataset-specific visual attribute directions to the query representation. This design is motivated by CLIP-guided image editing literature. StyleCLIP shows that meaningful semantic edit directions can be derived from CLIP text embeddings, including input-agnostic global directions computed from differences between target and neutral text prompts, and then used to manipulate images through vector operations in a latent space (Patashnik et al., 2021). HairCLIP provides a related precedent for using CLIP to combine multiple conditions, such as hairstyle and hair color specified by text or reference images, in order to perform targeted attribute editing while preserving irrelevant attributes (Wei et al., 2022). Although these works focus on image generation and editing, they support the core idea used here: CLIP embeddings can provide semantic directions, and text and visual conditions can be combined rather than treated as separate retrieval signals.

In our retrieval setting, no generator, mapper, or additional neural network is trained. Instead, we adapt the CLIP-guided editing idea directly in embedding space. For each query, the text direction is computed from the signed positive and negative CLIP attribute embeddings, while the visual direction is computed from CelebA image-embedding statistics for the same attributes.

The implementation uses four hybrid-family gates: `alpha` is the reference image weight, `beta_text` is the text-side edit weight, `beta_visual_pos` is the positive visual-direction weight, and `beta_visual_neg` is the negative visual-direction weight. In mathematical notation these are written as $\alpha$, $\beta_{\text{text}}$, $\beta_{\text{visual}}^{+}$, and $\beta_{\text{visual}}^{-}$, respectively. The final hybrid query is:

$$
\mathbf{q}
=
\operatorname{normalize}
\left(
\alpha \mathbf{v}_{\text{ref}}
+
\beta_{\text{text}} \boldsymbol{\Delta}_{\text{txt}}
+
\beta_{\text{visual}}^{+} \boldsymbol{\Delta}_{\text{vis}}^{+}
-
\beta_{\text{visual}}^{-} \boldsymbol{\Delta}_{\text{vis}}^{-}
\right).
$$

The text-side direction is:

$$
\boldsymbol{\Delta}_{\text{txt}}
=
\sum_{p \in \mathcal{P}} \mathbf{t}_p
-
\sum_{n \in \mathcal{N}} \mathbf{t}_n.
$$

The visual components are:

$$
\boldsymbol{\Delta}_{\text{vis}}^{+}
=
\sum_{p \in \mathcal{P}} \mathbf{d}_p,
\qquad
\boldsymbol{\Delta}_{\text{vis}}^{-}
=
\sum_{n \in \mathcal{N}} \mathbf{d}_n.
$$

Here $\boldsymbol{\Delta}_{\text{txt}}$ already carries its own internal positive/negative split; the text contribution is therefore controlled by the single implementation gate `beta_text`.

The fusion weights control how strongly the query preserves the reference image, follows generic CLIP text semantics, and follows dataset-specific visual attribute evidence. This is the main methodological contribution of the project: the retrieval query combines CLIP's language-guided edit signal with visual directions estimated from the target dataset, then ranks gallery images by cosine similarity after L2 normalization.


### 2.4.2. Combined text and visual fusion with prompt ensembling

Prompt ensembling was tested as a refinement of the text component in the combined fusion model. Instead of representing each CelebA attribute with a single prompt, we construct an embedding ensemble before inserting the resulting text vector into the hybrid query.

Prompt ensembling modifies only how each per-attribute text vector $\mathbf{t}_a$ entering $\boldsymbol{\Delta}_{\text{txt}}$ is computed; the fusion formulas above are unchanged. Let $c_1, \dots, c_{12}$ denote the $12$ carrier templates and $\text{embed}(\cdot)$ the CLIP text encoder. We evaluate two constructions.

1. *Ensemble of positive prompts* — the mean of the $M$ positive-prompt embeddings:

$$
\mathbf{t}_a^{\text{ens}} = \frac{\boldsymbol{e}_a}{\lVert \boldsymbol{e}_a \rVert_2},
\qquad
\boldsymbol{e}_a = \frac{1}{M} \sum_{i=1}^{M} \text{embed}\!\left(c_i(a)\right).
$$

This is motivated by CLIP’s original zero-shot classification procedure, where Radford et al. (2021) average predictions across many prompt templates to reduce sensitivity to wording and obtain a more stable text representation.

In particular, we average 12 prompt phrases: "a photo of a person {phrase}", "a close-up photo of a person {phrase}", "a headshot of a person {phrase}", "a portrait of someone {phrase}", "a cropped photo of a person {phrase}", "a good photo of a person {phrase}", "a bad photo of a person {phrase}", "a photo of the face of a person {phrase}", "a photo of a celebrity {phrase}", "a high quality photo of a person {phrase}", "a low quality photo of a person {phrase}", "an image of a person {phrase}". Instead of {phrase}, 40 CelebA attributes were inserted after a transformation to produce a meaningful sentence (e.g. "5_o_Clock_Shadow" --> "with a five o'clock shadow").

2. *Difference ensemble* — the normalized difference between the mean positive-prompt embedding and the mean neutral-prompt embedding:

$$
\mathbf{t}_a^{\text{diff}} = \frac{\boldsymbol{e}_a^{+} - \boldsymbol{e}_a^{0}}{\lVert \boldsymbol{e}_a^{+} - \boldsymbol{e}_a^{0} \rVert_2},
\qquad
\boldsymbol{e}_a^{+} = \frac{1}{M}\sum_{i=1}^{M}\text{embed}\!\left(c_i^{+}(a)\right),
\quad
\boldsymbol{e}_a^{0} = \frac{1}{M}\sum_{i=1}^{M}\text{embed}\!\left(c_i^{0}\right),
$$

where $c_i^{+}(a)$ slots attribute $a$ into carrier $i$ and $c_i^{0}$ is the corresponding neutral (attribute-free) carrier. The neutral carriers contain no negation; this avoids CLIP's poor handling of negated text, which would otherwise make $\boldsymbol{e}_a^{+} - \boldsymbol{e}_a^{0}$ collapse toward zero.

This construction inspired by StyleCLIP, where Patashnik et al. (2021) compute global edit directions from the difference between target and neutral text embeddings. In particular, we used 5 positive prompts (with an attribute present): "a photo of a face that is {modification_text}", "a portrait photo of a person who is {modification_text}", "a close-up face image of a person who is {modification_text}", "a celebrity face photo with {modification_text}", "a human face that is {modification_text}"; and 5 corresponding neutral prompts: "a photo of a face", "a portrait photo of a person", "a close-up face image", "a celebrity face photo", "a human face".

The goal of prompt ensembling is therefore not to change the retrieval architecture or the hybrid-family gates (`alpha`, `beta_text`, `beta_visual_pos`, and `beta_visual_neg`), but to make $\boldsymbol{\Delta}_{\text{txt}}$ less dependent on a single hand-written phrase and more robust as the text-side edit signal used in the combined text-and-visual query.


## 2.5. Reliability-weighted text and visual fusion

The hybrid text-and-visual fusion method combines three sources of information: the reference image embedding, the CLIP text-side edit direction, and CelebA visual attribute directions. However, not all visual attribute directions are equally reliable. Some attributes produce a clearer separation in the CLIP visual embedding space, while others are noisier or more ambiguous.

For this reason, this variant introduces a training-free reliability score for each CelebA attribute. The score is computed from the CelebA train split using the frozen CLIP image embeddings. It measures how well the positive and negative examples of an attribute are separated relative to their within-class spread.

Let \( \mathbf{v}_i \in \mathbb{R}^{512} \) be the CLIP image embedding of image \(i\), and let \(y_{i,a} \in \{-1,1\}\) be the CelebA label of image \(i\) for attribute \(a\). We define the positive and negative sets for attribute \(a\) as:

$$
\mathcal{I}_a^{+}
=
\{i : y_{i,a} = 1\},
\qquad
\mathcal{I}_a^{-}
=
\{i : y_{i,a} = -1\}.
$$

The positive and negative mean embeddings are:

$$
\boldsymbol{\mu}_a^{+}
=
\frac{1}{|\mathcal{I}_a^{+}|}
\sum_{i \in \mathcal{I}_a^{+}} \mathbf{v}_i,
\qquad
\boldsymbol{\mu}_a^{-}
=
\frac{1}{|\mathcal{I}_a^{-}|}
\sum_{i \in \mathcal{I}_a^{-}} \mathbf{v}_i.
$$

The raw visual direction for attribute \(a\) is computed as the difference between the positive and negative means:

$$
\mathbf{d}_a^{\text{raw}}
=
\boldsymbol{\mu}_a^{+}
-
\boldsymbol{\mu}_a^{-}.
$$

The normalized visual direction used for retrieval is:

$$
\mathbf{d}_a
=
\operatorname{normalize}
\left(
\mathbf{d}_a^{\text{raw}}
\right).
$$

To estimate how reliable this direction is, we compute the average spread of the positive and negative examples around their corresponding means:

$$
s_a^{+}
=
\frac{1}{|\mathcal{I}_a^{+}|}
\sum_{i \in \mathcal{I}_a^{+}}
\left\|
\mathbf{v}_i - \boldsymbol{\mu}_a^{+}
\right\|_2,
$$

$$
s_a^{-}
=
\frac{1}{|\mathcal{I}_a^{-}|}
\sum_{i \in \mathcal{I}_a^{-}}
\left\|
\mathbf{v}_i - \boldsymbol{\mu}_a^{-}
\right\|_2.
$$

The raw reliability score is then defined as the distance between the positive and negative means divided by the total within-class spread:

$$
\hat{r}_a
=
\frac{
\left\|
\boldsymbol{\mu}_a^{+}
-
\boldsymbol{\mu}_a^{-}
\right\|_2
}{
s_a^{+} + s_a^{-} + \epsilon
}.
$$

This score is high when the positive and negative examples of an attribute are well separated and compact in CLIP space. It is low when the attribute direction is weak or noisy relative to the spread of the corresponding examples.

The raw reliability values are min-max normalized across all CelebA attributes:

$$
r_a
=
\frac{
\hat{r}_a - \min_{b \in \mathcal{A}} \hat{r}_b
}{
\max_{b \in \mathcal{A}} \hat{r}_b
-
\min_{b \in \mathcal{A}} \hat{r}_b
}.
$$

where \(\mathcal{A}\) is the set of CelebA attributes. Therefore, \(r_a \in [0,1]\), with higher values assigned to visually more separable attributes.

The text-side modification direction is kept unchanged from the hybrid fusion method:

$$
\boldsymbol{\Delta}_{\text{txt}}
=
\sum_{p \in \mathcal{P}} \mathbf{t}_p
-
\sum_{n \in \mathcal{N}} \mathbf{t}_n.
$$

The reliability-weighted positive and negative visual components are:

$$
\boldsymbol{\Delta}_{\text{vis}}^{+}
=
\sum_{p \in \mathcal{P}} r_p \mathbf{d}_p,
\qquad
\boldsymbol{\Delta}_{\text{vis}}^{-}
=
\sum_{n \in \mathcal{N}} r_n \mathbf{d}_n.
$$

The final reliability-weighted hybrid query uses the same implementation gates as the hybrid-family methods: `alpha` for the reference image, `beta_text` for the text-side edit, `beta_visual_pos` for the positive visual direction, and `beta_visual_neg` for the negative visual direction.

$$
\mathbf{q}
=
\operatorname{normalize}
\left(
\alpha \mathbf{v}_{\text{ref}}
+
\beta_{\text{text}} \boldsymbol{\Delta}_{\text{txt}}
+
\beta_{\text{visual}}^{+} \boldsymbol{\Delta}_{\text{vis}}^{+}
-
\beta_{\text{visual}}^{-} \boldsymbol{\Delta}_{\text{vis}}^{-}
\right).
$$

If all reliability scores are set to \(r_a = 1\), this method reduces to the original hybrid text-and-visual fusion model. Therefore, the only difference introduced by this variant is that visual directions with stronger train-split separability receive more weight than visually noisy directions.


## 2.5.1. Direction-decorrelation reliability-weighted hybrid fusion

The reliability-weighted hybrid method assumes that each CelebA visual direction $\mathbf{d}_a$ represents an independent semantic edit. In practice, this assumption is not always valid. Some CelebA attributes are visually entangled: for example, `Male`, `Mustache`, `Heavy Makeup`, `Wearing Lipstick`, and `Young` may be aligned or anti-aligned in the CLIP embedding space. As a result, the visual direction of one attribute may partially encode another attribute.

To reduce this effect, this variant applies a training-free direction decorrelation step directly in CLIP space before using the visual directions in the hybrid query. The implementation selects confounding directions using absolute cosine similarity between visual directions, rather than label-correlation statistics.

Let $\mathbf{d}_a \in \mathbb{R}^{512}$ be the visual direction associated with attribute $a$. For two attributes $a$ and $b$, we measure the alignment between their visual directions using cosine similarity:

$$
\rho(a,b) =
\frac{\mathbf{d}_a^\top \mathbf{d}_b}
{\lVert \mathbf{d}_a \rVert_2 \lVert \mathbf{d}_b \rVert_2}.
$$

A high absolute value of $\rho(a,b)$ means that the two directions are strongly aligned or anti-aligned in the CLIP visual embedding space. Therefore, changing one attribute may also unintentionally move the query along another attribute direction.

For each attribute $a$, the implementation selects potentially confounding directions as:

$$
\mathcal{C}(a)
=
\{b \neq a : |\rho(a,b)| \geq \tau\}.
$$

In practice, only the top $M$ directions with the highest absolute cosine similarity are kept, in order to avoid over-correcting the attribute direction.

Let the selected confounders for attribute $a$ be $b_1, b_2, \dots, b_m$. We build a matrix $\mathbf{C}_a$ whose columns are the normalized confounder directions:

$$
\mathbf{C}_a
=
\left[
\frac{\mathbf{d}_{b_1}}{\lVert \mathbf{d}_{b_1} \rVert_2},
\frac{\mathbf{d}_{b_2}}{\lVert \mathbf{d}_{b_2} \rVert_2},
\dots,
\frac{\mathbf{d}_{b_m}}{\lVert \mathbf{d}_{b_m} \rVert_2}
\right].
$$

It then removes the component of $\mathbf{d}_a$ lying in the subspace spanned by the selected confounder directions using ridge-regularized projection:

$$
\mathbf{c}_a
=
(\mathbf{C}_a^\top \mathbf{C}_a + \lambda \mathbf{I})^{-1}
\mathbf{C}_a^\top \mathbf{d}_a.
$$

$$
\tilde{\mathbf{d}}_a
=
\mathbf{d}_a - \mathbf{C}_a \mathbf{c}_a.
$$

Finally, the direction is normalized again:

$$
\tilde{\mathbf{d}}_a
\leftarrow
\frac{\tilde{\mathbf{d}}_a}{\lVert \tilde{\mathbf{d}}_a \rVert_2}.
$$

The text-side modification direction remains the same as in the hybrid fusion method:

$$
\boldsymbol{\Delta}_{\text{txt}}
=
\sum_{p \in \mathcal{P}} \mathbf{t}_p
-
\sum_{n \in \mathcal{N}} \mathbf{t}_n.
$$

The reliability-weighted visual components are now built from the decorrelated directions:

$$
\boldsymbol{\Delta}_{\text{vis}}^{+}
=
\sum_{p \in \mathcal{P}} r_p \tilde{\mathbf{d}}_p,
\qquad
\boldsymbol{\Delta}_{\text{vis}}^{-}
=
\sum_{n \in \mathcal{N}} r_n \tilde{\mathbf{d}}_n.
$$

The final query uses the same hybrid-family gate notation:

$$
\mathbf{q}
=
\operatorname{normalize}
\left(
\alpha \mathbf{v}_{\text{ref}}
+
\beta_{\text{text}} \boldsymbol{\Delta}_{\text{txt}}
+
\beta_{\text{visual}}^{+} \boldsymbol{\Delta}_{\text{vis}}^{+}
-
\beta_{\text{visual}}^{-} \boldsymbol{\Delta}_{\text{vis}}^{-}
\right).
$$

Here, $r_a$ is the separation-based reliability score for attribute $a$. If all $r_a = 1$ and $\tilde{\mathbf{d}}_a = \mathbf{d}_a$, the method reduces to the original hybrid text-and-visual fusion model.

This method remains training-free. It does not learn a new neural model; it only modifies the visual-direction table before retrieval by reducing overlap between highly aligned or anti-aligned attribute directions in CLIP space.


# 3. Experiments and results

All hyperparameters were selected on the CelebA validation split and evaluated once on the official CelebA test split. The validation split contains 19,867 images and the test split contains 19,962 images. For each method, we report `Recall@1`, `Recall@5`, `Recall@10`, `Precision@1`, `Precision@5`, and `Precision@10` averaged across all valid source images in the corresponding ground-truth JSON.

The validation split is used only for grid search over fusion weights. The test split is kept held out and used only for the final reported metrics.

## 3.1. Baseline

### Baseline functions definitions

In [ ]:
# baseline retrieval helper functions
def parse_query_string(query: str) -> tuple[list[str], list[str]]:
    """Split a signed query string into positive and negative attributes."""
    positive = []
    negative = []
    for part in query.split(","):
        part = part.strip()
        if part.startswith("+"):
            positive.append(part[1:].strip())
        elif part.startswith("-"):
            negative.append(part[1:].strip())
    return positive, negative


def attribute_to_prompt(attribute: str) -> str:
    """Convert a CelebA attribute name into the saved CLIP text prompt."""
    return f"a photo of a person who is {attribute.replace('_', ' ').lower()}"


def build_query_embeddings(
    reference_embeddings: torch.Tensor,
    positive_embeddings: torch.Tensor | None,
    negative_embeddings: torch.Tensor | None,
    alpha: float = 1.0,
    beta: float = 1.0,
) -> torch.Tensor:
    """Apply image + positive text - negative text arithmetic fusion."""
    queries = reference_embeddings.float()
    if positive_embeddings is not None and positive_embeddings.numel() > 0:
        queries = queries + alpha * positive_embeddings.float().sum(dim=0, keepdim=True)
    if negative_embeddings is not None and negative_embeddings.numel() > 0:
        queries = queries - beta * negative_embeddings.float().sum(dim=0, keepdim=True)
    return l2_normalize(queries)


def retrieve_top_k_batch(
    query_embeddings: torch.Tensor,
    image_embeddings: torch.Tensor,
    image_ids: list[int],
    k: int,
    exclude_ids: list[int],
    device: str = "cpu",
) -> list[list[tuple[int, float]]]:
    """Retrieve top K image IDs for each query embedding."""
    query_embeddings = query_embeddings.to(device).float()
    image_embeddings = image_embeddings.to(device).float()
    similarities = query_embeddings @ image_embeddings.T

    id_to_row = {int(image_id): row for row, image_id in enumerate(image_ids)}
    for query_row, image_id in enumerate(exclude_ids):
        row = id_to_row.get(int(image_id))
        if row is not None:
            similarities[query_row, row] = -torch.inf

    values, indices = torch.topk(similarities, k=k, dim=1)
    results = []
    for row in range(query_embeddings.shape[0]):
        results.append(
            [
                (int(image_ids[index]), float(score))
                for index, score in zip(indices[row].cpu().tolist(), values[row].cpu())
            ]
        )
    return results


def load_ground_truth(path: str | Path) -> list[dict[str, Any]]:
    """Load a non-empty query/ground-truth JSON file."""
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty ground-truth file: {path}")
    with path.open("r", encoding="utf-8") as handle:
        data = json.load(handle)
    if not isinstance(data, list):
        raise ValueError("Ground-truth JSON must contain a list of query records.")
    return data


def run_baseline(
    query_json: str | Path,
    image_embedding_dir: str | Path,
    text_embedding_dir: str | Path,
    output_dir: str | Path | None = None,
    alpha: float = 1.0,
    beta: float = 1.0,
    top_k: tuple[int, ...] = (1, 5, 10),
    batch_size: int = 256,
    device: str = "cpu",
    save_metrics: bool = True,
    save_predictions: bool = False,
    save_run_config: bool = False,
) -> dict[str, Any]:
    """Run the arithmetic baseline, optionally saving metrics/predictions/config to output_dir."""
    if save_metrics or save_predictions or save_run_config:
        if output_dir is None:
            raise ValueError("output_dir is required when save_metrics, save_predictions, or save_run_config is True.")
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    image_embeddings, image_ids = load_embeddings(image_embedding_dir, "image_embeddings", "cpu")
    text_embeddings, text_ids = load_embeddings(text_embedding_dir, "text_embeddings", "cpu")
    image_ids = [int(image_id) for image_id in image_ids]
    image_row = {image_id: row for row, image_id in enumerate(image_ids)}
    text_lookup = {str(prompt): text_embeddings[row] for row, prompt in enumerate(text_ids)}
    entries = load_ground_truth(query_json)

    max_k = max(top_k)
    metric_rows = []
    prediction_rows = [] if save_predictions else None

    for entry in entries:
        positive, negative = parse_query_string(entry["query"])
        positive_embeddings = torch.stack(
            [text_lookup[attribute_to_prompt(attribute)] for attribute in positive]
        ) if positive else None
        negative_embeddings = torch.stack(
            [text_lookup[attribute_to_prompt(attribute)] for attribute in negative]
        ) if negative else None

        reference_items = [(int(reference), targets) for reference, targets in entry["ground_truth"].items()]
        for start in range(0, len(reference_items), batch_size):
            batch = reference_items[start:start + batch_size]
            references = [reference for reference, _ in batch]
            reference_rows = [image_row[reference] for reference in references]
            reference_embeddings = image_embeddings[reference_rows]
            query_embeddings = build_query_embeddings(
                reference_embeddings,
                positive_embeddings,
                negative_embeddings,
                alpha=alpha,
                beta=beta,
            )
            rankings = retrieve_top_k_batch(
                query_embeddings,
                image_embeddings,
                image_ids,
                k=max_k,
                exclude_ids=references,
                device=device,
            )

            for (reference, targets), ranking in zip(batch, rankings):
                retrieved = [image_id for image_id, _ in ranking]
                metrics = evaluate_single_ranking(retrieved, targets, ks=top_k)
                metric_rows.append(metrics)

                if prediction_rows is not None:
                    scores = [score for _, score in ranking]
                    prediction_rows.append(
                        {
                            "query": entry["query"],
                            "reference_index": reference,
                            "target_indices": " ".join(str(int(target)) for target in targets),
                            "retrieved_indices": " ".join(str(index) for index in retrieved),
                            "similarity_scores": " ".join(f"{score:.6f}" for score in scores),
                            **metrics,
                        }
                    )

    metrics = average_metrics(metric_rows)
    result = {
        "method": "clip_arithmetic_baseline",
        "query_json": str(query_json),
        "image_embedding_dir": str(image_embedding_dir),
        "text_embedding_dir": str(text_embedding_dir),
        "alpha": alpha,
        "beta": beta,
        "top_k": list(top_k),
        "query_instances": len(metric_rows),
        "gallery_size": len(image_ids),
        "metrics": metrics,
    }

    if save_metrics:
        (output_dir / "metrics.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    if prediction_rows is not None:
        with (output_dir / "predictions.csv").open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(prediction_rows[0].keys()))
            writer.writeheader()
            writer.writerows(prediction_rows)
    if save_run_config:
        (output_dir / "run_config.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    return result

### Baseline results

Simple arithmetic logic of the baseline performs quite poorly on the official test set. Specifically, it achieves `recall@1=0.024`, `recall@5=0.075` and `recall@10=0.116`.

In [ ]:
# preparing variables for the baseline run

# these parameters are for the baseline model
alpha = 1.0
beta = 1.0

baseline_output_dir = TMP_OUTPUT_ROOT / "baseline_run" / f"test_subset_alpha{alpha:.2f}_beta{beta:.2f}"

print(device)


cpu


In [ ]:
baseline_result = run_baseline(
    query_json=baseline_test_query_json,
    image_embedding_dir=baseline_img_embedding_test_dir,
    text_embedding_dir=baseline_txt_embedding_dir,
    output_dir=baseline_output_dir,
    alpha=alpha,
    beta=beta,
    batch_size=batch_size,
    device=device,
    save_predictions=save_predictions,
    save_run_config=save_run_config,
)

pd.DataFrame([baseline_result["metrics"]])

,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,0.024114,0.024114,0.075487,0.018879,0.116241,0.016474


## 3.2. Gated fusion grid search

### Gated fusion results

Since we are using `recall@10` as the basis for selecting the best results, the best gated fusion validation performance is achieved with $\alpha = \beta = 2$, assuming the same weight of positive and negative conditions. Interestingly though, slightly better `recall@1` is achieved with $\beta$ lower than $\alpha$ (e.g. $\alpha = 2$ and $\beta = 1$ or $0.5$), thus favoring positive conditions as more important.

This configuration achieves `recall@1=0.028`, `recall@5=0.09` and `recall@10=0.138` on the test split.

In [ ]:
# The alpha/beta grid search runs on the validation split; the test split
# stays held out for the final report.
alphas = GRID_VALUES
betas = GRID_VALUES

grid_results = []
for grid_alpha, grid_beta in itertools.product(alphas, betas):
    grid_result = run_baseline(
        query_json=baseline_val_query_json,
        image_embedding_dir=baseline_img_embedding_val_dir,
        text_embedding_dir=baseline_txt_embedding_dir,
        alpha=grid_alpha,
        beta=grid_beta,
        batch_size=batch_size,
        device=device,
        save_metrics=False,
        save_predictions=False,
        save_run_config=False,
    )
    grid_results.append({"alpha": grid_alpha, "beta": grid_beta, **grid_result["metrics"]})

grid_summary = pd.DataFrame(grid_results)
grid_summary.sort_values("recall@10", ascending=False).reset_index(drop=True).head(5)


,alpha,beta,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,2.0,2.00,0.027993,0.027993,0.093042,0.023017,0.143052,0.019812
1,2.0,1.50,0.027993,0.027993,0.093292,0.023134,0.142107,0.019737
2,2.0,1.00,0.028243,0.028243,0.093292,0.023117,0.141745,0.019643
3,2.0,0.50,0.028327,0.028327,0.092264,0.022839,0.140939,0.019545
4,2.0,0.25,0.028077,0.028077,0.091708,0.022645,0.139355,0.019353


In [ ]:
# The (alpha, beta) pair selected on validation is evaluated once on the
# held-out test split
best_grid_search = grid_summary.sort_values("recall@10", ascending=False).iloc[0]

gs_test_result = run_baseline(
    query_json=baseline_test_query_json,
    image_embedding_dir=baseline_img_embedding_test_dir,
    text_embedding_dir=baseline_txt_embedding_dir,
    alpha=float(best_grid_search["alpha"]),
    beta=float(best_grid_search["beta"]),
    batch_size=batch_size,
    device=device,
    save_metrics=False,
    save_predictions=False,
    save_run_config=False,
)

pd.DataFrame([{
    "alpha": float(best_grid_search["alpha"]),
    "beta": float(best_grid_search["beta"]),
    **gs_test_result["metrics"],
}])

,alpha,beta,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,2.0,2.0,0.028107,0.028107,0.089586,0.022147,0.138237,0.019218


We can see that modifying positive and negative condition weights improves the test performance in comparison with the baseline.

In [ ]:
comparison = pd.DataFrame([
    {
        "method": "baseline",
        "gamma": 1.0,
        "alpha": 1.0,
        "beta": 1.0,
        "recall@1": baseline_result["metrics"]["recall@1"],
        "recall@5": baseline_result["metrics"]["recall@5"],
        "recall@10": baseline_result["metrics"]["recall@10"],
        "precision@10": baseline_result["metrics"]["precision@10"],
    },
    {
        "method": "gated fusion grid search",
        "gamma": 1.0,
        "alpha": float(best_grid_search["alpha"]),
        "beta": float(best_grid_search["beta"]),
        "recall@1": gs_test_result["metrics"]["recall@1"],
        "recall@5": gs_test_result["metrics"]["recall@5"],
        "recall@10": gs_test_result["metrics"]["recall@10"],
        "precision@10":gs_test_result["metrics"]["precision@10"],
    },
])
comparison

,method,gamma,alpha,beta,recall@1,recall@5,recall@10,precision@10
0,baseline,1.0,1.0,1.0,0.024114,0.075487,0.116241,0.016474
1,gated fusion grid search,1.0,2.0,2.0,0.028107,0.089586,0.138237,0.019218


## 3.3. Visual-Direction Retrieval

### Visual direction functions definitions

In [ ]:
# visual-direction retrieval helper functions
# Gate names follow the section 3.3 formula: gamma weights the reference image,
# alpha the positive visual directions, beta the negative ones.
def build_visual_direction_queries(
    reference_embeddings: torch.Tensor,
    positive_directions: torch.Tensor | None,
    negative_directions: torch.Tensor | None,
    gamma: float = 1.0,
    alpha: float = 1.0,
    beta: float = 1.0,
) -> torch.Tensor:
    """Build normalized gamma * image + alpha * positive - beta * negative queries."""
    queries = gamma * reference_embeddings.float()
    if positive_directions is not None and positive_directions.numel() > 0:
        queries = queries + alpha * positive_directions.float().sum(dim=0, keepdim=True)
    if negative_directions is not None and negative_directions.numel() > 0:
        queries = queries - beta * negative_directions.float().sum(dim=0, keepdim=True)
    return l2_normalize(queries)


def run_visual_direction_retrieval(
    query_json: str | Path,
    image_embedding_dir: str | Path,
    visual_direction_dir: str | Path,
    output_dir: str | Path | None = None,
    gamma: float = 1.0,
    alpha: float = 1.0,
    beta: float = 1.0,
    top_k: tuple[int, ...] = (1, 5, 10),
    batch_size: int = 256,
    device: str = "cpu",
    save_metrics: bool = True,
    save_predictions: bool = False,
    save_run_config: bool = False,
) -> dict[str, Any]:
    """Run retrieval with precomputed visual attribute directions."""

    if save_metrics or save_predictions or save_run_config:
        if output_dir is None:
            raise ValueError("output_dir is required when save_metrics, save_predictions, or save_run_config is True.")
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    image_embeddings, image_ids = load_embeddings(image_embedding_dir, "image_embeddings", "cpu")
    direction_embeddings, direction_ids = load_embeddings(visual_direction_dir, "visual_directions", "cpu")
    image_ids = [int(image_id) for image_id in image_ids]
    image_row = {image_id: row for row, image_id in enumerate(image_ids)}
    # Direction ids are raw attribute names (e.g. "Smiling"), not prompt strings.
    direction_lookup = {str(attribute): direction_embeddings[row] for row, attribute in enumerate(direction_ids)}
    entries = load_ground_truth(query_json)

    max_k = max(top_k)
    metric_rows = []
    prediction_rows = [] if save_predictions else None

    for entry in entries:
        positive, negative = parse_query_string(entry["query"])
        positive_directions = torch.stack(
            [direction_lookup[attribute] for attribute in positive]
        ) if positive else None
        negative_directions = torch.stack(
            [direction_lookup[attribute] for attribute in negative]
        ) if negative else None

        reference_items = [(int(reference), targets) for reference, targets in entry["ground_truth"].items()]
        for start in range(0, len(reference_items), batch_size):
            batch = reference_items[start:start + batch_size]
            references = [reference for reference, _ in batch]
            reference_rows = [image_row[reference] for reference in references]
            reference_embeddings = image_embeddings[reference_rows]
            query_embeddings = build_visual_direction_queries(
                reference_embeddings,
                positive_directions,
                negative_directions,
                gamma=gamma,
                alpha=alpha,
                beta=beta,
            )
            rankings = retrieve_top_k_batch(
                query_embeddings,
                image_embeddings,
                image_ids,
                k=max_k,
                exclude_ids=references,
                device=device,
            )

            for (reference, targets), ranking in zip(batch, rankings):
                retrieved = [image_id for image_id, _ in ranking]
                metrics = evaluate_single_ranking(retrieved, targets, ks=top_k)
                metric_rows.append(metrics)

                if prediction_rows is not None:
                    scores = [score for _, score in ranking]
                    prediction_rows.append(
                        {
                            "query": entry["query"],
                            "reference_index": reference,
                            "target_indices": " ".join(str(int(target)) for target in targets),
                            "retrieved_indices": " ".join(str(index) for index in retrieved),
                            "similarity_scores": " ".join(f"{score:.6f}" for score in scores),
                            **metrics,
                        }
                    )

    metrics = average_metrics(metric_rows)
    result = {
        "method": "visual_direction_retrieval",
        "formula": "normalize(gamma * image + alpha * sum(positive directions) - beta * sum(negative directions))",
        "query_json": str(query_json),
        "image_embedding_dir": str(image_embedding_dir),
        "visual_direction_dir": str(visual_direction_dir),
        "gamma": gamma,
        "alpha": alpha,
        "beta": beta,
        "top_k": list(top_k),
        "query_instances": len(metric_rows),
        "gallery_size": len(image_ids),
        "metrics": metrics,
    }

    if save_metrics:
        (output_dir / "metrics.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    if prediction_rows is not None:
        with (output_dir / "predictions.csv").open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(prediction_rows[0].keys()))
            writer.writeheader()
            writer.writerows(prediction_rows)
    if save_run_config:
        (output_dir / "run_config.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    return result

def compute_visual_directions(
    image_embeddings: torch.Tensor,
    labels: torch.Tensor,
    attribute_names: list[str],
) -> tuple[torch.Tensor, list[dict[str, Any]]]:
    """Compute normalize(mean positive - mean negative) for each attribute."""
    embeddings = image_embeddings.float()
    directions = []
    diagnostics = []
    for column, attribute in enumerate(attribute_names):
        values = labels[:, column]
        pos_mask = values == 1
        neg_mask = values == -1
        n_pos = int(pos_mask.sum())
        n_neg = int(neg_mask.sum())
        if n_pos == 0 or n_neg == 0:
            raise ValueError(f"Attribute {attribute!r} needs both positive and negative examples.")
        pos_mean = embeddings[pos_mask].mean(dim=0)
        neg_mean = embeddings[neg_mask].mean(dim=0)
        raw_direction = pos_mean - neg_mean
        directions.append(l2_normalize(raw_direction))
        diagnostics.append(
            {
                "attribute": attribute,
                "support_positive": n_pos,
                "support_negative": n_neg,
                "balance_score": min(n_pos, n_neg) / max(n_pos, n_neg),
                "direction_norm_before_normalization": float(torch.linalg.norm(raw_direction)),
                "separation_score": float(1.0 - F.cosine_similarity(pos_mean.unsqueeze(0), neg_mean.unsqueeze(0))),
            }
        )
    return torch.stack(directions), diagnostics

### Visual direction computation

In [ ]:
# Download the official CelebA annotation files into data/celeba_full/ (idempotent:
# skipped when the files are already there). If the Google Drive quota blocks
# the download, fetch the same tables from the Kaggle mirror
# (jessicali9530/celeba-dataset) and place list_attr_celeba.csv and
# list_eval_partition.csv in data/celeba_full/; the loader below accepts both formats.
attr_txt_path = CELEBA_ROOT / "list_attr_celeba.txt"
partition_txt_path = CELEBA_ROOT / "list_eval_partition.txt"
attr_csv_path = CELEBA_ROOT / "list_attr_celeba.csv"
partition_csv_path = CELEBA_ROOT / "list_eval_partition.csv"

have_attr = attr_txt_path.exists() or attr_csv_path.exists()
have_partition = partition_txt_path.exists() or partition_csv_path.exists()

if not (have_attr and have_partition):

    # File ids of the official CelebA Google Drive folder (same ids torchvision uses).
    if not have_attr:
        gdown.download(id="0B7EVK8r0v71pblRyaVFSWGxPY0U", output=str(attr_txt_path), quiet=False)
    if not have_partition:
        gdown.download(id="0B7EVK8r0v71pY0NSMzRuSXJEVkk", output=str(partition_txt_path), quiet=False)

print("attributes:", attr_txt_path if attr_txt_path.exists() else attr_csv_path)
print("partition: ", partition_txt_path if partition_txt_path.exists() else partition_csv_path)


attributes: /content/data/celeba_full/list_attr_celeba.txt
partition:  /content/data/celeba_full/list_eval_partition.txt


In [ ]:
# MD5 checksums of the official .txt files, as published in torchvision's
# CelebA dataset class. A matching digest means the file is byte-identical to
# the canonical annotation table. Only checked for the .txt variant; the
# Kaggle .csv re-encoding has different bytes but identical content.
KNOWN_MD5 = {
    "list_attr_celeba.txt": "75e246fa4810816ffd6ee81facbd244c",
    "list_eval_partition.txt": "d32c9cbf5e040fd4025c592c306e6668",
}
for annotation_path in (attr_txt_path, partition_txt_path):
    if annotation_path.exists():
        digest = hashlib.md5(annotation_path.read_bytes()).hexdigest()
        assert digest == KNOWN_MD5[annotation_path.name], f"unexpected md5 for {annotation_path.name}: {digest}"
        print(f"md5 {annotation_path.name}: {digest} [OK]")

if attr_txt_path.exists():
    # Official format: line 1 is the row count, line 2 the 40 attribute names,
    # then one whitespace-separated row per image. The header has one field
    # fewer than the data rows, so pandas uses the filename column as index.
    attr_table = pd.read_csv(attr_txt_path, sep=r"\s+", skiprows=1)
else:
    attr_table = pd.read_csv(attr_csv_path, index_col=0)

if partition_txt_path.exists():
    partition = pd.read_csv(partition_txt_path, sep=r"\s+", header=None, index_col=0).squeeze("columns")
else:
    partition = pd.read_csv(partition_csv_path, index_col=0).squeeze("columns")

attr_table = attr_table.sort_index()
partition = partition.sort_index()
attribute_names = list(attr_table.columns)

assert attr_table.shape == (202599, 40), attr_table.shape
assert attr_table.isin([-1, 1]).all().all()
assert (attr_table.index == partition.index).all()
assert partition.value_counts().to_dict() == {0: 162770, 1: 19867, 2: 19962}

# The official partition is contiguous in filename order: train is exactly
# 000001.jpg .. 162770.jpg, so row i of the train slice pairs with row i of
# the full train embeddings.
train_filenames = attr_table.index[partition == 0]
assert list(train_filenames) == [f"{i:06d}.jpg" for i in range(1, 162771)]

print(f"all checks passed: 202,599 rows, {len(attribute_names)} attributes, "
      f"official split sizes 162,770 / 19,867 / 19,962")

md5 list_attr_celeba.txt: 75e246fa4810816ffd6ee81facbd244c [OK]
md5 list_eval_partition.txt: d32c9cbf5e040fd4025c592c306e6668 [OK]
all checks passed: 202,599 rows, 40 attributes, official split sizes 162,770 / 19,867 / 19,962


In [ ]:
# compute and save
full_train_embeddings, full_train_ids = load_embeddings(
    baseline_img_embedding_train_dir, "image_embeddings", "cpu"
)
assert [int(image_id) for image_id in full_train_ids] == list(range(162770))

train_labels = torch.tensor(attr_table.loc[partition == 0].to_numpy(), dtype=torch.int64)
full_visual_directions, full_direction_diagnostics = compute_visual_directions(
    full_train_embeddings, train_labels, attribute_names
)

full_direction_dir = EMBEDDING_ROOT
torch.save(full_visual_directions, full_direction_dir / "visual_directions.pt")
np.save(full_direction_dir / "visual_directions_ids.npy", np.array(attribute_names))


### Visual-direction grid search results


Grid search using visual directions instead of textual directions selected the best validation configuration at $\gamma=1.5$, $\alpha=1.0$, and $\beta=0.5$. On the held-out test split, this configuration achieves `Recall@1 = 0.030`, `Recall@5 = 0.096`, and `Recall@10 = 0.147`.

Interestingly, the selected visual-direction configuration assigns a larger weight to the reference image and a lower weight to negative conditions than to positive conditions.

In [ ]:
# In-memory grid search over the three gates, same pattern as the 3.2 sweep.
# The visual directions were precomputed from the train split and stored in
# data/celeba_full/embeddings/visual_directions.pt. The sweep runs on the
# validation split; the test split stays held out for the final report.
gamma_values = GRID_VALUES
alpha_values = GRID_VALUES
beta_values = GRID_VALUES

visual_grid_results = []
for grid_gamma, grid_alpha, grid_beta in itertools.product(gamma_values, alpha_values, beta_values):
    visual_result = run_visual_direction_retrieval(
        query_json=baseline_val_query_json,
        image_embedding_dir=baseline_img_embedding_val_dir,
        visual_direction_dir=visual_direction_dir,
        gamma=grid_gamma,
        alpha=grid_alpha,
        beta=grid_beta,
        batch_size=batch_size,
        device=device,
        save_metrics=False,
        save_predictions=False,
        save_run_config=False,
    )
    visual_grid_results.append(
        {"gamma": grid_gamma, "alpha": grid_alpha, "beta": grid_beta, **visual_result["metrics"]}
    )

visual_grid_summary = pd.DataFrame(visual_grid_results)
visual_grid_summary.sort_values("recall@10", ascending=False).reset_index(drop=True).head(10)


,gamma,alpha,beta,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,1.5,1.00,0.50,0.031301,0.031301,0.102132,0.025675,0.157702,0.022636
1,2.0,1.00,0.50,0.031023,0.031023,0.102605,0.025981,0.156812,0.022575
2,1.0,0.50,0.25,0.031023,0.031023,0.102605,0.025981,0.156812,0.022575
3,1.5,1.00,0.25,0.030328,0.030328,0.100381,0.025386,0.154449,0.022297
4,2.0,1.00,0.25,0.030467,0.030467,0.099853,0.025419,0.153365,0.022244
5,1.0,0.50,0.50,0.031357,0.031357,0.099797,0.025402,0.151975,0.022072
6,0.5,0.25,0.25,0.031357,0.031357,0.099797,0.025402,0.151975,0.022072
7,2.0,1.00,1.00,0.031357,0.031357,0.099797,0.025402,0.151975,0.022072
8,2.0,1.50,0.50,0.030134,0.030134,0.098602,0.024719,0.150780,0.021766
9,2.0,1.50,1.00,0.031051,0.031051,0.097545,0.024580,0.148778,0.021616


In [ ]:
# The (gamma, alpha, beta) triple selected on validation is evaluated once on
# the held-out test split
best_visual = visual_grid_summary.sort_values("recall@10", ascending=False).iloc[0]

visual_test_result = run_visual_direction_retrieval(
    query_json=baseline_test_query_json,
    image_embedding_dir=baseline_img_embedding_test_dir,
    visual_direction_dir=visual_direction_dir,
    gamma=float(best_visual["gamma"]),
    alpha=float(best_visual["alpha"]),
    beta=float(best_visual["beta"]),
    batch_size=batch_size,
    device=device,
    save_metrics=False,
    save_predictions=False,
    save_run_config=False,
)

pd.DataFrame([{
    "gamma": float(best_visual["gamma"]),
    "alpha": float(best_visual["alpha"]),
    "beta": float(best_visual["beta"]),
    **visual_test_result["metrics"],
}])

,gamma,alpha,beta,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,1.5,1.0,0.5,0.030165,0.030165,0.096273,0.023962,0.147374,0.021394


Thus, gated fusion with visual directions improves test performance compared with both the CLIP arithmetic baseline and the best text-only gated configuration.

In [ ]:
# Test-split metrics of each method's validation-selected configuration.
# baseline_result come from section 3.1,
# best_grid_search / gs_test_result come from section 3.2,
# best_visual / visual_test_result from section 3.3.
comparison = pd.DataFrame([
        {
        "method": "baseline",
        "gamma": 1.0,
        "alpha": 1.0,
        "beta": 1.0,
        "recall@1": baseline_result["metrics"]["recall@1"],
        "recall@5": baseline_result["metrics"]["recall@5"],
        "recall@10": baseline_result["metrics"]["recall@10"],
        "precision@10": baseline_result["metrics"]["precision@10"],
    },
       {
        "method": "gated fusion grid search with textual directions",
        "gamma": 1.0,
        "alpha": float(best_grid_search["alpha"]),
        "beta": float(best_grid_search["beta"]),
        "recall@1": gs_test_result["metrics"]["recall@1"],
        "recall@5": gs_test_result["metrics"]["recall@5"],
        "recall@10": gs_test_result["metrics"]["recall@10"],
        "precision@10":gs_test_result["metrics"]["precision@10"],
    },
    {
        "method": "gated fusion grid search with visual directions",
        "gamma": float(best_visual["gamma"]),
        "alpha": float(best_visual["alpha"]),
        "beta": float(best_visual["beta"]),
        "recall@1": visual_test_result["metrics"]["recall@1"],
        "recall@5": visual_test_result["metrics"]["recall@5"],
        "recall@10": visual_test_result["metrics"]["recall@10"],
        "precision@10": visual_test_result["metrics"]["precision@10"],
    },
])
comparison

,method,gamma,alpha,beta,recall@1,recall@5,recall@10,precision@10
0,baseline,1.0,1.0,1.0,0.024114,0.075487,0.116241,0.016474
1,gated fusion grid search with textual directions,1.0,2.0,2.0,0.028107,0.089586,0.138237,0.019218
2,gated fusion grid search with visual directions,1.5,1.0,0.5,0.030165,0.096273,0.147374,0.021394


## 3.4. Combined Text + Visual Fusion

### Hybrid fusion functions definitions



In [ ]:
# hybrid text + visual retrieval helper functions
# Gate names follow the section 3.4 formula: alpha weights the reference image,
# beta_text weights the CLIP text delta, beta_visual_pos the positive visual
# directions, and beta_visual_neg the negative visual directions.
def build_hybrid_queries(
    reference_embeddings: torch.Tensor,
    positive_text_embeddings: torch.Tensor | None,
    negative_text_embeddings: torch.Tensor | None,
    positive_visual_directions: torch.Tensor | None,
    negative_visual_directions: torch.Tensor | None,
    alpha: float = 1.0,
    beta_text: float = 1.0,
    beta_visual_pos: float = 0.5,
    beta_visual_neg: float = 0.25,
) -> torch.Tensor:
    """Build normalized image + text delta + signed visual correction queries."""
    queries = alpha * reference_embeddings.float()

    text_delta = torch.zeros_like(queries[:1])
    if positive_text_embeddings is not None and positive_text_embeddings.numel() > 0:
        text_delta = text_delta + positive_text_embeddings.float().sum(dim=0, keepdim=True)
    if negative_text_embeddings is not None and negative_text_embeddings.numel() > 0:
        text_delta = text_delta - negative_text_embeddings.float().sum(dim=0, keepdim=True)

    if positive_visual_directions is not None and positive_visual_directions.numel() > 0:
        queries = queries + beta_visual_pos * positive_visual_directions.float().sum(dim=0, keepdim=True)
    if negative_visual_directions is not None and negative_visual_directions.numel() > 0:
        queries = queries - beta_visual_neg * negative_visual_directions.float().sum(dim=0, keepdim=True)

    queries = queries + beta_text * text_delta
    return l2_normalize(queries)

def run_hybrid_fusion_retrieval(
    query_json: str | Path,
    image_embedding_dir: str | Path,
    text_embedding_dir: str | Path,
    visual_direction_dir: str | Path,
    output_dir: str | Path | None = None,
    text_embedding_name: str = "text_embeddings",
    alpha: float = 1.0,
    beta_text: float = 1.0,
    beta_visual_pos: float = 0.5,
    beta_visual_neg: float = 0.25,
    top_k: tuple[int, ...] = (1, 5, 10),
    batch_size: int = 256,
    device: str = "cpu",
    save_metrics: bool = True,
    save_predictions: bool = False,
    save_run_config: bool = False,
) -> dict[str, Any]:
    """Run query-level hybrid retrieval with CLIP text and visual directions."""
    if save_metrics or save_predictions or save_run_config:
        if output_dir is None:
            raise ValueError("output_dir is required when save_metrics, save_predictions, or save_run_config is True.")
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    image_embeddings, image_ids = load_embeddings(image_embedding_dir, "image_embeddings", "cpu")
    text_embeddings, text_ids = load_embeddings(text_embedding_dir, text_embedding_name, "cpu")
    visual_directions, visual_ids = load_embeddings(visual_direction_dir, "visual_directions", "cpu")

    image_ids = [int(image_id) for image_id in image_ids]
    text_ids = [str(prompt) for prompt in text_ids]
    visual_ids = [str(attribute) for attribute in visual_ids]

    image_row = {image_id: row for row, image_id in enumerate(image_ids)}
    text_lookup = {prompt: text_embeddings[row] for row, prompt in enumerate(text_ids)}
    visual_lookup = {attribute: visual_directions[row] for row, attribute in enumerate(visual_ids)}
    entries = load_ground_truth(query_json)

    max_k = max(top_k)
    metric_rows = []
    prediction_rows = [] if save_predictions else None

    for entry in entries:
        positive, negative = parse_query_string(entry["query"])

        positive_text = torch.stack(
            [text_lookup[attribute_to_prompt(attribute)] for attribute in positive]
        ) if positive else None
        negative_text = torch.stack(
            [text_lookup[attribute_to_prompt(attribute)] for attribute in negative]
        ) if negative else None

        positive_visual = torch.stack(
            [visual_lookup[attribute] for attribute in positive]
        ) if positive else None
        negative_visual = torch.stack(
            [visual_lookup[attribute] for attribute in negative]
        ) if negative else None

        reference_items = [(int(reference), targets) for reference, targets in entry["ground_truth"].items()]
        for start in range(0, len(reference_items), batch_size):
            batch = reference_items[start:start + batch_size]
            references = [reference for reference, _ in batch]
            reference_rows = [image_row[reference] for reference in references]
            reference_embeddings = image_embeddings[reference_rows]

            query_embeddings = build_hybrid_queries(
                reference_embeddings=reference_embeddings,
                positive_text_embeddings=positive_text,
                negative_text_embeddings=negative_text,
                positive_visual_directions=positive_visual,
                negative_visual_directions=negative_visual,
                alpha=alpha,
                beta_text=beta_text,
                beta_visual_pos=beta_visual_pos,
                beta_visual_neg=beta_visual_neg,
            )

            rankings = retrieve_top_k_batch(
                query_embeddings=query_embeddings,
                image_embeddings=image_embeddings,
                image_ids=image_ids,
                k=max_k,
                exclude_ids=references,
                device=device,
            )

            for (reference, targets), ranking in zip(batch, rankings):
                retrieved = [image_id for image_id, _ in ranking]
                metrics = evaluate_single_ranking(retrieved, targets, ks=top_k)
                metric_rows.append(metrics)

                if prediction_rows is not None:
                    scores = [score for _, score in ranking]
                    prediction_rows.append(
                        {
                            "query": entry["query"],
                            "reference_index": reference,
                            "target_indices": " ".join(str(int(target)) for target in targets),
                            "retrieved_indices": " ".join(str(index) for index in retrieved),
                            "similarity_scores": " ".join(f"{score:.6f}" for score in scores),
                            **metrics,
                        }
                    )

    metrics = average_metrics(metric_rows)
    result = {
        "method": "hybrid_text_visual_fusion",
        "formula": (
            "normalize(alpha * image + beta_text * (sum(text_pos) - sum(text_neg)) "
            "+ beta_visual_pos * sum(visual_pos) - beta_visual_neg * sum(visual_neg))"
        ),
        "query_json": str(query_json),
        "image_embedding_dir": str(image_embedding_dir),
        "text_embedding_dir": str(text_embedding_dir),
        "text_embedding_name": text_embedding_name,
        "visual_direction_dir": str(visual_direction_dir),
        "alpha": alpha,
        "beta_text": beta_text,
        "beta_visual_pos": beta_visual_pos,
        "beta_visual_neg": beta_visual_neg,
        "top_k": list(top_k),
        "query_instances": len(metric_rows),
        "gallery_size": len(image_ids),
        "metrics": metrics,
    }

    if save_metrics:
        (output_dir / "metrics.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")

    if prediction_rows is not None:
        with (output_dir / "predictions.csv").open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(prediction_rows[0].keys()))
            writer.writeheader()
            writer.writerows(prediction_rows)

    if save_run_config:
        (output_dir / "run_config.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")

    return result


### Hybrid fusion grid search results


In [ ]:
# In-memory grid search over the four hybrid gates, same pattern as the 3.2 and 3.3 sweeps.
# Text embeddings and visual directions were precomputed and stored in
# data/celeba_full/embeddings/. The sweep runs on the validation split; the
# test split stays held out for the final report.
alpha_values = GRID_VALUES
beta_text_values = GRID_VALUES
beta_visual_pos_values = GRID_VALUES
beta_visual_neg_values = GRID_VALUES

hybrid_grid_results = []
for grid_alpha, grid_beta_text, grid_beta_visual_pos, grid_beta_visual_neg in itertools.product(
    alpha_values,
    beta_text_values,
    beta_visual_pos_values,
    beta_visual_neg_values,
):
    hybrid_result = run_hybrid_fusion_retrieval(
        query_json=baseline_val_query_json,
        image_embedding_dir=baseline_img_embedding_val_dir,
        text_embedding_dir=hybrid_text_embedding_dir,
        visual_direction_dir=hybrid_visual_direction_dir,
        alpha=grid_alpha,
        beta_text=grid_beta_text,
        beta_visual_pos=grid_beta_visual_pos,
        beta_visual_neg=grid_beta_visual_neg,
        batch_size=batch_size,
        device=device,
        save_metrics=False,
        save_predictions=False,
        save_run_config=False,
    )
    hybrid_grid_results.append({
        "alpha": grid_alpha,
        "beta_text": grid_beta_text,
        "beta_visual_pos": grid_beta_visual_pos,
        "beta_visual_neg": grid_beta_visual_neg,
        **hybrid_result["metrics"],
    })

hybrid_grid_summary = pd.DataFrame(hybrid_grid_results)
hybrid_grid_summary.sort_values("recall@10", ascending=False).reset_index(drop=True).head(10)


,alpha,beta_text,beta_visual_pos,beta_visual_neg,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,1.0,2.0,0.25,0.25,0.034220,0.034220,0.112501,0.028043,0.174603,0.024385
1,1.0,2.0,0.50,0.25,0.033720,0.033720,0.110583,0.027632,0.172685,0.024224
2,1.5,2.0,0.50,0.50,0.033887,0.033887,0.111500,0.027865,0.172351,0.024555
3,1.0,1.5,0.50,0.25,0.034331,0.034331,0.112084,0.028160,0.171740,0.024596
4,1.5,2.0,0.50,0.25,0.033136,0.033136,0.111056,0.027821,0.170378,0.024343
5,1.0,1.0,0.50,0.25,0.034081,0.034081,0.111917,0.028088,0.170239,0.024499
6,2.0,2.0,1.00,0.50,0.034081,0.034081,0.111917,0.028088,0.170239,0.024499
7,1.0,1.5,0.25,0.25,0.032886,0.032886,0.109888,0.027459,0.169961,0.024043
8,2.0,2.0,1.00,0.25,0.033553,0.033553,0.110666,0.027993,0.168765,0.024310
9,2.0,1.5,1.00,0.50,0.033664,0.033664,0.110666,0.027771,0.168237,0.024299


In [ ]:
# The (alpha, beta_text, beta_visual_pos, beta_visual_neg) tuple selected on
# validation is evaluated once on the held-out test split.
best_hybrid = hybrid_grid_summary.sort_values("recall@10", ascending=False).iloc[0]

hybrid_test_result = run_hybrid_fusion_retrieval(
    query_json=baseline_test_query_json,
    image_embedding_dir=baseline_img_embedding_test_dir,
    text_embedding_dir=hybrid_text_embedding_dir,
    visual_direction_dir=hybrid_visual_direction_dir,
    alpha=float(best_hybrid["alpha"]),
    beta_text=float(best_hybrid["beta_text"]),
    beta_visual_pos=float(best_hybrid["beta_visual_pos"]),
    beta_visual_neg=float(best_hybrid["beta_visual_neg"]),
    batch_size=batch_size,
    device=device,
    save_metrics=False,
    save_predictions=False,
    save_run_config=False,
)

pd.DataFrame([{
    "alpha": float(best_hybrid["alpha"]),
    "beta_text": float(best_hybrid["beta_text"]),
    "beta_visual_pos": float(best_hybrid["beta_visual_pos"]),
    "beta_visual_neg": float(best_hybrid["beta_visual_neg"]),
    **hybrid_test_result["metrics"],
}])

,alpha,beta_text,beta_visual_pos,beta_visual_neg,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,1.0,2.0,0.25,0.25,0.033099,0.033099,0.106469,0.026613,0.164317,0.023291


In [ ]:

hybrid_comparison = pd.DataFrame([
    {
        "method": "baseline",
        "alpha": 1.0,
        "beta_text": 1.0,
        "beta_visual_pos": None,
        "beta_visual_neg": None,
        "recall@1": baseline_result["metrics"]["recall@1"],
        "recall@5": baseline_result["metrics"]["recall@5"],
        "recall@10": baseline_result["metrics"]["recall@10"],
        "precision@10": baseline_result["metrics"]["precision@10"],
    },
    {
        "method": "gated fusion with textual directions",
        "alpha": float(best_grid_search["alpha"]),
        "beta_text": float(best_grid_search["beta"]),
        "beta_visual_pos": None,
        "beta_visual_neg": None,
        "recall@1": gs_test_result["metrics"]["recall@1"],
        "recall@5": gs_test_result["metrics"]["recall@5"],
        "recall@10": gs_test_result["metrics"]["recall@10"],
        "precision@10": gs_test_result["metrics"]["precision@10"],
    },
    {
        "method": "gated fusion with visual directions",
        "alpha": float(best_visual["gamma"]),
        "beta_text": None,
        "beta_visual_pos": float(best_visual["alpha"]),
        "beta_visual_neg": float(best_visual["beta"]),
        "recall@1": visual_test_result["metrics"]["recall@1"],
        "recall@5": visual_test_result["metrics"]["recall@5"],
        "recall@10": visual_test_result["metrics"]["recall@10"],
        "precision@10": visual_test_result["metrics"]["precision@10"],
    },
    {
        "method": "hybrid text + visual fusion",
        "alpha": float(best_hybrid["alpha"]),
        "beta_text": float(best_hybrid["beta_text"]),
        "beta_visual_pos": float(best_hybrid["beta_visual_pos"]),
        "beta_visual_neg": float(best_hybrid["beta_visual_neg"]),
        "recall@1": hybrid_test_result["metrics"]["recall@1"],
        "recall@5": hybrid_test_result["metrics"]["recall@5"],
        "recall@10": hybrid_test_result["metrics"]["recall@10"],
        "precision@10": hybrid_test_result["metrics"]["precision@10"],
    },
])

hybrid_comparison

,method,alpha,beta_text,beta_visual_pos,beta_visual_neg,recall@1,recall@5,recall@10,precision@10
0,baseline,1.0,1.0,NaN,NaN,0.024114,0.075487,0.116241,0.016474
1,gated fusion with textual directions,2.0,2.0,NaN,NaN,0.028107,0.089586,0.138237,0.019218
2,gated fusion with visual directions,1.5,NaN,1.00,0.50,0.030165,0.096273,0.147374,0.021394
3,hybrid text + visual fusion,1.0,2.0,0.25,0.25,0.033099,0.106469,0.164317,0.023291


### 3.4.1. Prompt-Ensemble Hybrid Check

### Prompt-ensemble text embedding functions
To keep the section lightweight, the retrieval logic is shared by both prompt averaging and prompt differencing; only the text delta construction changes.

In [ ]:
# prompt-text helper functions
# Prompt averaging stores one text vector per CelebA attribute.
# Prompt differencing stores one neutral-subtracted text vector per full query.
PROMPT_AVERAGE_CARRIERS = [
    "a photo of a person {phrase}",
    "a close-up photo of a person {phrase}",
    "a headshot of a person {phrase}",
    "a portrait of someone {phrase}",
    "a cropped photo of a person {phrase}",
    "a good photo of a person {phrase}",
    "a bad photo of a person {phrase}",
    "a photo of the face of a person {phrase}",
    "a photo of a celebrity {phrase}",
    "a high quality photo of a person {phrase}",
    "a low quality photo of a person {phrase}",
    "an image of a person {phrase}",
]

PROMPT_DIFFERENCE_POSITIVE_CARRIERS = [
    "a photo of a face that is {modification_text}",
    "a portrait photo of a person who is {modification_text}",
    "a close-up face image of a person who is {modification_text}",
    "a celebrity face photo with {modification_text}",
    "a human face that is {modification_text}",
]

PROMPT_DIFFERENCE_NEUTRAL_CARRIERS = [
    "a photo of a face",
    "a portrait photo of a person",
    "a close-up face image",
    "a celebrity face photo",
    "a human face",
]

ATTRIBUTE_PHRASES = {
    "5_o_Clock_Shadow": "with a five o'clock shadow",
    "Arched_Eyebrows": "with arched eyebrows",
    "Attractive": "who is attractive",
    "Bags_Under_Eyes": "with bags under their eyes",
    "Bald": "who is bald",
    "Bangs": "with bangs",
    "Big_Lips": "with big lips",
    "Big_Nose": "with a big nose",
    "Black_Hair": "with black hair",
    "Blond_Hair": "with blond hair",
    "Blurry": "who looks blurry",
    "Brown_Hair": "with brown hair",
    "Bushy_Eyebrows": "with bushy eyebrows",
    "Chubby": "who is chubby",
    "Double_Chin": "with a double chin",
    "Eyeglasses": "with eyeglasses",
    "Goatee": "with a goatee",
    "Gray_Hair": "with gray hair",
    "Heavy_Makeup": "wearing heavy makeup",
    "High_Cheekbones": "with high cheekbones",
    "Male": "who is a man",
    "Mouth_Slightly_Open": "with their mouth slightly open",
    "Mustache": "with a mustache",
    "Narrow_Eyes": "with narrow eyes",
    "No_Beard": "who is smooth-faced",
    "Oval_Face": "with an oval face",
    "Pale_Skin": "with pale skin",
    "Pointy_Nose": "with a pointy nose",
    "Receding_Hairline": "with a receding hairline",
    "Rosy_Cheeks": "with rosy cheeks",
    "Sideburns": "with sideburns",
    "Smiling": "who is smiling",
    "Straight_Hair": "with straight hair",
    "Wavy_Hair": "with wavy hair",
    "Wearing_Earrings": "wearing earrings",
    "Wearing_Hat": "wearing a hat",
    "Wearing_Lipstick": "wearing lipstick",
    "Wearing_Necklace": "wearing a necklace",
    "Wearing_Necktie": "wearing a necktie",
    "Young": "who is young",
}


def encode_text_prompt_batch(prompts: list[str], model, processor, device: torch.device) -> torch.Tensor:
    """Encode a text prompt batch with CLIP and normalize each row."""
    inputs = processor(
        text=prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77,
    ).to(device)
    with torch.no_grad():
        features = model.get_text_features(**inputs)
        features = getattr(features, "pooler_output", features)
    return l2_normalize(features.cpu())


def build_prompt_average_prompts(attribute: str) -> list[str]:
    """Return positive prompt variants for one CelebA attribute."""
    phrase = ATTRIBUTE_PHRASES[attribute]
    return [carrier.format(phrase=phrase) for carrier in PROMPT_AVERAGE_CARRIERS]


def attribute_to_modification_phrase(attribute: str, positive: bool = True) -> str:
    """Convert a signed CelebA attribute into readable prompt text."""
    phrase = ATTRIBUTE_PHRASES[attribute]
    phrase = phrase.replace("with ", "having ")
    phrase = phrase.replace("who is ", "")
    return phrase if positive else "not " + phrase


def query_to_modification_text(query: str) -> str:
    """Convert a signed query string into one modification phrase."""
    positive, negative = parse_query_string(query)
    parts = [attribute_to_modification_phrase(attribute, True) for attribute in positive]
    parts.extend(attribute_to_modification_phrase(attribute, False) for attribute in negative)
    return " and ".join(parts)


def unique_queries_from_files(paths: list[Path]) -> list[str]:
    """Return unique query strings from one or more query JSON files."""
    seen = set()
    queries = []
    for path in paths:
        for entry in load_ground_truth(path):
            query = str(entry["query"])
            if query not in seen:
                seen.add(query)
                queries.append(query)
    return queries


def build_prompt_average_embeddings(output_dir: str | Path, model_name: str, batch_size: int, device: str) -> None:
    """Build one averaged text embedding per CelebA attribute."""

    output_dir = Path(output_dir)
    attributes = list(ATTRIBUTE_PHRASES.keys())
    prompts = []
    spans = []
    for attribute in attributes:
        start = len(prompts)
        prompts.extend(build_prompt_average_prompts(attribute))
        spans.append((start, len(prompts)))

    resolved_device = torch.device(device)
    processor = CLIPProcessor.from_pretrained(model_name)
    model = CLIPModel.from_pretrained(model_name).to(resolved_device)
    model.eval()

    rows = []
    for start in tqdm(range(0, len(prompts), batch_size), desc="Encoding prompt average"):
        rows.append(encode_text_prompt_batch(prompts[start:start + batch_size], model, processor, resolved_device))

    prompt_embeddings = torch.cat(rows, dim=0)
    averaged = torch.stack([
        l2_normalize(prompt_embeddings[start:end].mean(dim=0))
        for start, end in spans
    ])
    ids = [attribute_to_prompt(attribute) for attribute in attributes]

    if torch.isnan(averaged).any():
        raise ValueError("Prompt-averaged embeddings contain NaN values.")
    torch.save(averaged.float(), output_dir / "prompt_average_text_embeddings.pt")
    np.save(output_dir / "prompt_average_text_embeddings_ids.npy", np.array(ids))


def build_prompt_difference_embeddings(query_paths: list[Path], output_dir: str | Path, model_name: str, device: str) -> None:
    """Build one neutral-subtracted text embedding per unique signed query."""

    output_dir = Path(output_dir)
    queries = unique_queries_from_files(query_paths)
    resolved_device = torch.device(device)
    processor = CLIPProcessor.from_pretrained(model_name)
    model = CLIPModel.from_pretrained(model_name).to(resolved_device)
    model.eval()

    neutral_embeddings = encode_text_prompt_batch(
        PROMPT_DIFFERENCE_NEUTRAL_CARRIERS,
        model,
        processor,
        resolved_device,
    )
    neutral_mean = neutral_embeddings.mean(dim=0)

    rows = []
    for query in tqdm(queries, desc="Encoding prompt differences"):
        modification_text = query_to_modification_text(query)
        prompts = [
            carrier.format(modification_text=modification_text)
            for carrier in PROMPT_DIFFERENCE_POSITIVE_CARRIERS
        ]
        positive_mean = encode_text_prompt_batch(prompts, model, processor, resolved_device).mean(dim=0)
        rows.append(l2_normalize(positive_mean - neutral_mean))

    embeddings = torch.stack(rows)
    if torch.isnan(embeddings).any():
        raise ValueError("Prompt-difference embeddings contain NaN values.")
    torch.save(embeddings.float(), output_dir / "prompt_difference_text_embeddings.pt")
    np.save(output_dir / "prompt_difference_text_embeddings_ids.npy", np.array(queries))


def build_prompt_text_queries(
    reference_embeddings: torch.Tensor,
    text_delta: torch.Tensor,
    positive_visual_directions: torch.Tensor | None,
    negative_visual_directions: torch.Tensor | None,
    alpha: float = 1.0,
    beta_text: float = 1.0,
    beta_visual_pos: float = 0.5,
    beta_visual_neg: float = 0.25,
) -> torch.Tensor:
    """Build normalized hybrid queries from image, prompt text, and visual directions."""
    queries = alpha * reference_embeddings.float()
    queries = queries + beta_text * text_delta.float().unsqueeze(0)
    if positive_visual_directions is not None and positive_visual_directions.numel() > 0:
        queries = queries + beta_visual_pos * positive_visual_directions.float().sum(dim=0, keepdim=True)
    if negative_visual_directions is not None and negative_visual_directions.numel() > 0:
        queries = queries - beta_visual_neg * negative_visual_directions.float().sum(dim=0, keepdim=True)
    return l2_normalize(queries)


def run_prompt_text_retrieval(
    query_json: str | Path,
    image_embedding_dir: str | Path,
    prompt_embedding_dir: str | Path,
    visual_direction_dir: str | Path,
    prompt_mode: str,
    text_embedding_name: str,
    alpha: float = 1.0,
    beta_text: float = 1.0,
    beta_visual_pos: float = 0.5,
    beta_visual_neg: float = 0.25,
    top_k: tuple[int, ...] = (1, 5, 10),
    batch_size: int = 256,
    device: str = "cpu",
) -> dict[str, Any]:
    """Run prompt-averaging or prompt-difference hybrid retrieval."""
    if prompt_mode not in {"averaging", "differencing"}:
        raise ValueError("prompt_mode must be 'averaging' or 'differencing'.")

    image_embeddings, image_ids = load_embeddings(image_embedding_dir, "image_embeddings", "cpu")
    text_embeddings, text_ids = load_embeddings(prompt_embedding_dir, text_embedding_name, "cpu")
    visual_directions, visual_ids = load_embeddings(visual_direction_dir, "visual_directions", "cpu")

    image_ids = [int(image_id) for image_id in image_ids]
    text_ids = [str(text_id) for text_id in text_ids]
    visual_ids = [str(attribute) for attribute in visual_ids]

    image_row = {image_id: row for row, image_id in enumerate(image_ids)}
    text_lookup = {text_id: text_embeddings[row] for row, text_id in enumerate(text_ids)}
    visual_lookup = {attribute: visual_directions[row] for row, attribute in enumerate(visual_ids)}
    entries = load_ground_truth(query_json)

    max_k = max(top_k)
    metric_rows = []

    for entry in entries:
        positive, negative = parse_query_string(entry["query"])
        if prompt_mode == "averaging":
            text_delta = torch.zeros_like(text_embeddings[0]).float()
            if positive:
                text_delta = text_delta + torch.stack([
                    text_lookup[attribute_to_prompt(attribute)] for attribute in positive
                ]).float().sum(dim=0)
            if negative:
                text_delta = text_delta - torch.stack([
                    text_lookup[attribute_to_prompt(attribute)] for attribute in negative
                ]).float().sum(dim=0)
        else:
            text_delta = text_lookup[str(entry["query"])]

        positive_visual = torch.stack([visual_lookup[attribute] for attribute in positive]) if positive else None
        negative_visual = torch.stack([visual_lookup[attribute] for attribute in negative]) if negative else None

        reference_items = [(int(reference), targets) for reference, targets in entry["ground_truth"].items()]
        for start in range(0, len(reference_items), batch_size):
            batch = reference_items[start:start + batch_size]
            references = [reference for reference, _ in batch]
            reference_rows = [image_row[reference] for reference in references]
            query_embeddings = build_prompt_text_queries(
                reference_embeddings=image_embeddings[reference_rows],
                text_delta=text_delta,
                positive_visual_directions=positive_visual,
                negative_visual_directions=negative_visual,
                alpha=alpha,
                beta_text=beta_text,
                beta_visual_pos=beta_visual_pos,
                beta_visual_neg=beta_visual_neg,
            )
            rankings = retrieve_top_k_batch(
                query_embeddings=query_embeddings,
                image_embeddings=image_embeddings,
                image_ids=image_ids,
                k=max_k,
                exclude_ids=references,
                device=device,
            )
            for (_, targets), ranking in zip(batch, rankings):
                retrieved = [image_id for image_id, _ in ranking]
                metric_rows.append(evaluate_single_ranking(retrieved, targets, ks=top_k))

    return {
        "method": f"prompt_{prompt_mode}_hybrid_fusion",
        "alpha": alpha,
        "beta_text": beta_text,
        "beta_visual_pos": beta_visual_pos,
        "beta_visual_neg": beta_visual_neg,
        "query_instances": len(metric_rows),
        "metrics": average_metrics(metric_rows),
    }


In [ ]:
# Prompt text embeddings are cached once and reused by both validation grid search and test evaluation.
prompt_model_name = CLIP_MODEL_NAME

RUN_PROMPT_AVERAGING_EXTRACTION = not (prompt_embedding_dir / "prompt_average_text_embeddings.pt").exists()
RUN_PROMPT_DIFFERENCE_EXTRACTION = not (prompt_embedding_dir / "prompt_difference_text_embeddings.pt").exists()

if RUN_PROMPT_AVERAGING_EXTRACTION:
    build_prompt_average_embeddings(
        output_dir=prompt_embedding_dir,
        model_name=prompt_model_name,
        batch_size=64,
        device=device,
    )

if RUN_PROMPT_DIFFERENCE_EXTRACTION:
    build_prompt_difference_embeddings(
        query_paths=[baseline_val_query_json, baseline_test_query_json],
        output_dir=prompt_embedding_dir,
        model_name=prompt_model_name,
        device=device,
    )

assert (prompt_embedding_dir / "prompt_average_text_embeddings.pt").exists()
assert (prompt_embedding_dir / "prompt_difference_text_embeddings.pt").exists()


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Encoding prompt differences:   0%|          | 0/13 [00:00<?, ?it/s]

### Prompt-averaging hybrid grid search results

The prompt-averaging hybrid keeps the same hybrid-family query form and gates: `alpha` weights the reference image, `beta_text` weights the text-side edit, `beta_visual_pos` weights the positive visual directions, and `beta_visual_neg` weights the negative visual directions. Only the per-attribute text vector changes, replacing $\mathbf{t}_a$ in

$$
\boldsymbol{\Delta}_{\text{txt}}
=
\sum_{p \in \mathcal{P}} \mathbf{t}_p
-
\sum_{n \in \mathcal{N}} \mathbf{t}_n
$$

with the prompt-averaged vector $\mathbf{t}_a^{\text{ens}}$. The final query therefore remains

$$
\mathbf{q}
=
\operatorname{normalize}
\left(
\alpha \mathbf{v}_{\text{ref}}
+
\beta_{\text{text}} \boldsymbol{\Delta}_{\text{txt}}
+
\beta_{\text{visual}}^{+} \boldsymbol{\Delta}_{\text{vis}}^{+}
-
\beta_{\text{visual}}^{-} \boldsymbol{\Delta}_{\text{vis}}^{-}
\right).
$$


In [ ]:
# In-memory grid search for the prompt-averaging hybrid, same pattern as the 3.4 sweep.
# The visual directions stay fixed; only the text embeddings are replaced by
# prompt-averaged attribute embeddings. The sweep runs on validation.
alpha_values = GRID_VALUES
beta_text_values = GRID_VALUES
beta_visual_pos_values = GRID_VALUES
beta_visual_neg_values = GRID_VALUES

prompt_average_grid_results = []

for grid_alpha, grid_beta_text, grid_beta_visual_pos, grid_beta_visual_neg in itertools.product(
    alpha_values,
    beta_text_values,
    beta_visual_pos_values,
    beta_visual_neg_values,
):
    prompt_average_result = run_prompt_text_retrieval(
        query_json=baseline_val_query_json,
        image_embedding_dir=baseline_img_embedding_val_dir,
        prompt_embedding_dir=prompt_embedding_dir,
        visual_direction_dir=hybrid_visual_direction_dir,
        prompt_mode="averaging",
        text_embedding_name="prompt_average_text_embeddings",
        alpha=grid_alpha,
        beta_text=grid_beta_text,
        beta_visual_pos=grid_beta_visual_pos,
        beta_visual_neg=grid_beta_visual_neg,
        batch_size=batch_size,
        device=device,
    )
    prompt_average_grid_results.append({
        "alpha": grid_alpha,
        "beta_text": grid_beta_text,
        "beta_visual_pos": grid_beta_visual_pos,
        "beta_visual_neg": grid_beta_visual_neg,
        **prompt_average_result["metrics"],
    })

prompt_average_grid_summary = pd.DataFrame(prompt_average_grid_results)
prompt_average_grid_summary.sort_values("recall@10", ascending=False).reset_index(drop=True).head(10)


,alpha,beta_text,beta_visual_pos,beta_visual_neg,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,1.0,2.0,0.25,0.25,0.036166,0.036166,0.118533,0.029311,0.179412,0.025224
1,1.0,1.5,0.25,0.25,0.034442,0.034442,0.114113,0.028444,0.175270,0.024816
2,1.5,2.0,0.50,0.25,0.034637,0.034637,0.116170,0.029094,0.174381,0.024933
3,1.5,2.0,0.50,0.50,0.034887,0.034887,0.115670,0.028905,0.174325,0.024896
4,1.0,2.0,0.50,0.25,0.033553,0.033553,0.111945,0.027843,0.172991,0.024307
5,1.0,1.5,0.50,0.25,0.033525,0.033525,0.113585,0.028521,0.172741,0.024630
6,2.0,2.0,1.00,0.50,0.033831,0.033831,0.112557,0.028360,0.171684,0.024730
7,1.0,1.0,0.50,0.25,0.033831,0.033831,0.112557,0.028360,0.171684,0.024730
8,2.0,2.0,1.00,0.25,0.033692,0.033692,0.112723,0.028494,0.170934,0.024674
9,1.5,1.5,0.50,0.50,0.033442,0.033442,0.112251,0.028082,0.170656,0.024429


In [ ]:
# The prompt-averaging gates selected on validation are evaluated once on
# the held-out test split.
best_prompt_average = prompt_average_grid_summary.sort_values("recall@10", ascending=False).iloc[0]

prompt_average_test_result = run_prompt_text_retrieval(
    query_json=baseline_test_query_json,
    image_embedding_dir=baseline_img_embedding_test_dir,
    prompt_embedding_dir=prompt_embedding_dir,
    visual_direction_dir=hybrid_visual_direction_dir,
    prompt_mode="averaging",
    text_embedding_name="prompt_average_text_embeddings",
    alpha=float(best_prompt_average["alpha"]),
    beta_text=float(best_prompt_average["beta_text"]),
    beta_visual_pos=float(best_prompt_average["beta_visual_pos"]),
    beta_visual_neg=float(best_prompt_average["beta_visual_neg"]),
    batch_size=batch_size,
    device=device,
)

pd.DataFrame([{
    "alpha": float(best_prompt_average["alpha"]),
    "beta_text": float(best_prompt_average["beta_text"]),
    "beta_visual_pos": float(best_prompt_average["beta_visual_pos"]),
    "beta_visual_neg": float(best_prompt_average["beta_visual_neg"]),
    **prompt_average_test_result["metrics"],
}])


,alpha,beta_text,beta_visual_pos,beta_visual_neg,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,1.0,2.0,0.25,0.25,0.033704,0.033704,0.109615,0.027163,0.167947,0.023862


### Prompt-difference hybrid grid search results

The prompt-difference variant replaces the summed attribute text embeddings with one neutral-subtracted text direction for the full signed query. In the shared hybrid formula, this query-level vector occupies $\boldsymbol{\Delta}_{\text{txt}}$, while the visual branch and implementation gates remain unchanged: `alpha`, `beta_text`, `beta_visual_pos`, and `beta_visual_neg`.

$$
\mathbf{q}
=
\operatorname{normalize}
\left(
\alpha \mathbf{v}_{\text{ref}}
+
\beta_{\text{text}} \boldsymbol{\Delta}_{\text{txt}}
+
\beta_{\text{visual}}^{+} \boldsymbol{\Delta}_{\text{vis}}^{+}
-
\beta_{\text{visual}}^{-} \boldsymbol{\Delta}_{\text{vis}}^{-}
\right).
$$


In [ ]:
# In-memory grid search for the prompt-difference hybrid, same pattern as the 3.4 sweep.
# The sweep runs on validation; the test split stays held out for the final report.

alpha_values = GRID_VALUES
beta_text_values = GRID_VALUES
beta_visual_pos_values = GRID_VALUES
beta_visual_neg_values = GRID_VALUES

prompt_difference_grid_results = []

for grid_alpha, grid_beta_text, grid_beta_visual_pos, grid_beta_visual_neg in itertools.product(
    alpha_values,
    beta_text_values,
    beta_visual_pos_values,
    beta_visual_neg_values,
):
    prompt_difference_result = run_prompt_text_retrieval(
        query_json=baseline_val_query_json,
        image_embedding_dir=baseline_img_embedding_val_dir,
        prompt_embedding_dir=prompt_embedding_dir,
        visual_direction_dir=hybrid_visual_direction_dir,
        prompt_mode="differencing",
        text_embedding_name="prompt_difference_text_embeddings",
        alpha=grid_alpha,
        beta_text=grid_beta_text,
        beta_visual_pos=grid_beta_visual_pos,
        beta_visual_neg=grid_beta_visual_neg,
        batch_size=batch_size,
        device=device,
    )
    prompt_difference_grid_results.append({
        "alpha": grid_alpha,
        "beta_text": grid_beta_text,
        "beta_visual_pos": grid_beta_visual_pos,
        "beta_visual_neg": grid_beta_visual_neg,
        **prompt_difference_result["metrics"],
    })

prompt_difference_grid_summary = pd.DataFrame(prompt_difference_grid_results)
prompt_difference_grid_summary.sort_values("recall@10", ascending=False).reset_index(drop=True).head(10)


,alpha,beta_text,beta_visual_pos,beta_visual_neg,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,1.0,1.0,0.25,0.50,0.036388,0.036388,0.119951,0.030301,0.182720,0.026331
1,2.0,2.0,0.50,1.00,0.036388,0.036388,0.119951,0.030301,0.182720,0.026331
2,1.5,2.0,0.25,0.50,0.035777,0.035777,0.117143,0.029411,0.182025,0.026000
3,2.0,2.0,0.25,1.00,0.035666,0.035666,0.118005,0.029828,0.181525,0.025994
4,1.0,1.5,0.25,0.50,0.036110,0.036110,0.114753,0.028749,0.181219,0.025533
5,1.5,1.5,0.25,0.50,0.034470,0.034470,0.116671,0.029706,0.180191,0.025914
6,1.5,1.5,0.50,0.50,0.035360,0.035360,0.117644,0.029761,0.179496,0.025883
7,1.0,1.0,0.25,0.25,0.035165,0.035165,0.116671,0.029672,0.178300,0.025775
8,2.0,2.0,0.50,0.50,0.035165,0.035165,0.116671,0.029672,0.178300,0.025775
9,1.5,2.0,0.50,0.50,0.035193,0.035193,0.114308,0.028727,0.178050,0.025411


In [ ]:
# The prompt-difference gates selected on validation are evaluated once on
# the held-out test split.
best_prompt_difference = prompt_difference_grid_summary.sort_values("recall@10", ascending=False).iloc[0]

prompt_difference_test_result = run_prompt_text_retrieval(
    query_json=baseline_test_query_json,
    image_embedding_dir=baseline_img_embedding_test_dir,
    prompt_embedding_dir=prompt_embedding_dir,
    visual_direction_dir=hybrid_visual_direction_dir,
    prompt_mode="differencing",
    text_embedding_name="prompt_difference_text_embeddings",
    alpha=float(best_prompt_difference["alpha"]),
    beta_text=float(best_prompt_difference["beta_text"]),
    beta_visual_pos=float(best_prompt_difference["beta_visual_pos"]),
    beta_visual_neg=float(best_prompt_difference["beta_visual_neg"]),
    batch_size=batch_size,
    device=device,
)

pd.DataFrame([{
    "alpha": float(best_prompt_difference["alpha"]),
    "beta_text": float(best_prompt_difference["beta_text"]),
    "beta_visual_pos": float(best_prompt_difference["beta_visual_pos"]),
    "beta_visual_neg": float(best_prompt_difference["beta_visual_neg"]),
    **prompt_difference_test_result["metrics"],
}])


,alpha,beta_text,beta_visual_pos,beta_visual_neg,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,1.0,1.0,0.25,0.5,0.036548,0.036548,0.113095,0.028555,0.173938,0.025018


### Prompt-Ensemble Takeaway

The prompt-based variants slightly improve over the original hybrid text-and-visual fusion. The original hybrid method reaches `Recall@10 = 0.164`, while prompt averaging reaches `Recall@10 = 0.168` and prompt differencing reaches `Recall@10 = 0.174`.

The prompt-difference variant is the strongest result in the notebook. This suggests that representing the requested edit as a difference between a modified query prompt and a neutral reference prompt provides a cleaner text-side direction than simply averaging several attribute prompts. However, the improvement remains modest, which supports the broader observation that fixed-weight fusion variants begin to saturate around 17% `Recall@10`.


## 3.5. Reliability weighted hybrid text and visual fusion
The reliability-weighted hybrid keeps the same text-and-visual fusion formula from section 3.4, but it scales each visual attribute direction by a train-split reliability score before adding it to the query. The score is high when the positive and negative image-embedding means for an attribute are well separated and low when the attribute direction is noisy relative to the within-class spread.

The expensive part is computing the reliability artifacts from the full CelebA train split. The cell below caches those tensors in `data/celeba_full/embeddings`, so later runs only reload them.


In [ ]:
# reliability-weighted hybrid helper functions
# This section reuses the full CelebA annotations loaded in section 3.3.
def save_embedding_files(embeddings: torch.Tensor, ids: list[Any], output_dir: str | Path, name: str) -> None:
    """Save an embedding tensor and matching row ids."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    torch.save(embeddings.cpu().float(), output_dir / f"{name}.pt")
    np.save(output_dir / f"{name}_ids.npy", np.array(ids))


def min_max_normalize(values: torch.Tensor) -> torch.Tensor:
    """Scale reliability values to [0, 1]."""
    denominator = values.max() - values.min()
    if float(denominator.item()) == 0.0:
        return torch.zeros_like(values)
    return (values - values.min()) / denominator


def compute_reliability_visual_artifacts(
    train_embeddings: torch.Tensor,
    train_labels: torch.Tensor,
    attribute_names: list[str],
    output_dir: str | Path,
    epsilon: float = 1e-12,
) -> None:
    """Compute train-split visual directions and reliability scores."""
    normalized_directions = []
    raw_directions = []
    raw_reliability_values = []

    for col, attribute in enumerate(attribute_names):
        values = train_labels[:, col]
        pos_embeddings = train_embeddings[values == 1].float()
        neg_embeddings = train_embeddings[values == -1].float()
        if pos_embeddings.numel() == 0 or neg_embeddings.numel() == 0:
            raise ValueError(f"Attribute {attribute!r} needs both positive and negative examples.")

        pos_mean = pos_embeddings.mean(dim=0)
        neg_mean = neg_embeddings.mean(dim=0)
        raw_direction = pos_mean - neg_mean
        spread_pos = torch.linalg.norm(pos_embeddings - pos_mean, dim=1).mean()
        spread_neg = torch.linalg.norm(neg_embeddings - neg_mean, dim=1).mean()
        raw_reliability = torch.linalg.norm(raw_direction) / (spread_pos + spread_neg + epsilon)

        normalized_directions.append(l2_normalize(raw_direction))
        raw_directions.append(raw_direction)
        raw_reliability_values.append(raw_reliability)

    reliability = min_max_normalize(torch.stack(raw_reliability_values).float())
    save_embedding_files(torch.stack(normalized_directions), attribute_names, output_dir, "visual_directions_reliable")
    save_embedding_files(torch.stack(raw_directions), attribute_names, output_dir, "visual_directions_raw")
    save_embedding_files(reliability.unsqueeze(1), attribute_names, output_dir, "visual_direction_reliability")


def build_reliability_hybrid_queries(
    reference_embeddings: torch.Tensor,
    positive_text_embeddings: torch.Tensor | None,
    negative_text_embeddings: torch.Tensor | None,
    positive_visual_directions: torch.Tensor | None,
    negative_visual_directions: torch.Tensor | None,
    positive_reliability: torch.Tensor | None,
    negative_reliability: torch.Tensor | None,
    alpha: float = 1.0,
    beta_text: float = 2.0,
    beta_visual_pos: float = 0.5,
    beta_visual_neg: float = 0.25,
) -> torch.Tensor:
    """Build normalized image + text delta + reliability-weighted visual correction queries."""
    queries = alpha * reference_embeddings.float()

    text_delta = torch.zeros_like(queries[:1])
    if positive_text_embeddings is not None and positive_text_embeddings.numel() > 0:
        text_delta = text_delta + positive_text_embeddings.float().sum(dim=0, keepdim=True)
    if negative_text_embeddings is not None and negative_text_embeddings.numel() > 0:
        text_delta = text_delta - negative_text_embeddings.float().sum(dim=0, keepdim=True)

    if positive_visual_directions is not None and positive_visual_directions.numel() > 0:
        weighted_positive = positive_visual_directions.float() * positive_reliability.float().view(-1, 1)
        queries = queries + beta_visual_pos * weighted_positive.sum(dim=0, keepdim=True)
    if negative_visual_directions is not None and negative_visual_directions.numel() > 0:
        weighted_negative = negative_visual_directions.float() * negative_reliability.float().view(-1, 1)
        queries = queries - beta_visual_neg * weighted_negative.sum(dim=0, keepdim=True)

    queries = queries + beta_text * text_delta
    return l2_normalize(queries)


def run_reliability_hybrid_retrieval(
    query_json: str | Path,
    image_embedding_dir: str | Path,
    text_embedding_dir: str | Path,
    visual_direction_dir: str | Path,
    text_embedding_name: str = "text_embeddings",
    visual_direction_name: str = "visual_directions_reliable",
    reliability_name: str = "visual_direction_reliability",
    alpha: float = 1.0,
    beta_text: float = 2.0,
    beta_visual_pos: float = 0.5,
    beta_visual_neg: float = 0.25,
    top_k: tuple[int, ...] = (1, 5, 10),
    batch_size: int = 256,
    device: str = "cpu",
) -> dict[str, Any]:
    """Run hybrid retrieval with visual directions scaled by reliability."""
    image_embeddings, image_ids = load_embeddings(image_embedding_dir, "image_embeddings", "cpu")
    text_embeddings, text_ids = load_embeddings(text_embedding_dir, text_embedding_name, "cpu")
    visual_directions, visual_ids = load_embeddings(visual_direction_dir, visual_direction_name, "cpu")
    reliability_scores, reliability_ids = load_embeddings(visual_direction_dir, reliability_name, "cpu")

    if reliability_scores.dim() == 2:
        reliability_scores = reliability_scores.squeeze(1)

    image_ids = [int(image_id) for image_id in image_ids]
    text_ids = [str(prompt) for prompt in text_ids]
    visual_ids = [str(attribute) for attribute in visual_ids]
    reliability_ids = [str(attribute) for attribute in reliability_ids]

    image_row = {image_id: row for row, image_id in enumerate(image_ids)}
    text_lookup = {prompt: text_embeddings[row] for row, prompt in enumerate(text_ids)}
    visual_lookup = {attribute: visual_directions[row] for row, attribute in enumerate(visual_ids)}
    reliability_lookup = {attribute: reliability_scores[row] for row, attribute in enumerate(reliability_ids)}
    entries = load_ground_truth(query_json)

    max_k = max(top_k)
    metric_rows = []
    for entry in entries:
        positive, negative = parse_query_string(entry["query"])
        positive_text = torch.stack([text_lookup[attribute_to_prompt(attribute)] for attribute in positive]) if positive else None
        negative_text = torch.stack([text_lookup[attribute_to_prompt(attribute)] for attribute in negative]) if negative else None
        positive_visual = torch.stack([visual_lookup[attribute] for attribute in positive]) if positive else None
        negative_visual = torch.stack([visual_lookup[attribute] for attribute in negative]) if negative else None
        positive_reliability = torch.stack([reliability_lookup[attribute] for attribute in positive]) if positive else None
        negative_reliability = torch.stack([reliability_lookup[attribute] for attribute in negative]) if negative else None

        reference_items = [(int(reference), targets) for reference, targets in entry["ground_truth"].items()]
        for start in range(0, len(reference_items), batch_size):
            batch = reference_items[start:start + batch_size]
            references = [reference for reference, _ in batch]
            reference_rows = [image_row[reference] for reference in references]
            query_embeddings = build_reliability_hybrid_queries(
                reference_embeddings=image_embeddings[reference_rows],
                positive_text_embeddings=positive_text,
                negative_text_embeddings=negative_text,
                positive_visual_directions=positive_visual,
                negative_visual_directions=negative_visual,
                positive_reliability=positive_reliability,
                negative_reliability=negative_reliability,
                alpha=alpha,
                beta_text=beta_text,
                beta_visual_pos=beta_visual_pos,
                beta_visual_neg=beta_visual_neg,
            )
            rankings = retrieve_top_k_batch(
                query_embeddings=query_embeddings,
                image_embeddings=image_embeddings,
                image_ids=image_ids,
                k=max_k,
                exclude_ids=references,
                device=device,
            )
            for (_, targets), ranking in zip(batch, rankings):
                retrieved = [image_id for image_id, _ in ranking]
                metric_rows.append(evaluate_single_ranking(retrieved, targets, ks=top_k))

    return {
        "method": "reliability_hybrid_text_visual_fusion",
        "alpha": alpha,
        "beta_text": beta_text,
        "beta_visual_pos": beta_visual_pos,
        "beta_visual_neg": beta_visual_neg,
        "query_instances": len(metric_rows),
        "metrics": average_metrics(metric_rows),
    }


In [ ]:
# Reliability artifacts are cached after the first computation.
if not (reliability_embedding_dir / "visual_direction_reliability.pt").exists():
    full_train_embeddings, full_train_ids = load_embeddings(reliability_train_embedding_dir, "image_embeddings", "cpu")
    assert [int(image_id) for image_id in full_train_ids] == list(range(len(full_train_ids)))

    # attr_table and partition are loaded in section 3.3 from the full CelebA annotation files.
    train_labels = torch.tensor(attr_table.loc[partition == 0].to_numpy(), dtype=torch.int64)
    assert train_labels.shape[0] == len(full_train_ids)

    compute_reliability_visual_artifacts(
        train_embeddings=full_train_embeddings,
        train_labels=train_labels,
        attribute_names=attribute_names,
        output_dir=reliability_embedding_dir,
    )

assert (reliability_embedding_dir / "visual_directions_reliable.pt").exists()
assert (reliability_embedding_dir / "visual_direction_reliability.pt").exists()


In [ ]:
# Reliability-weighted hybrid grid search over the same fixed gate grid used by the hybrid methods.
reliability_alpha_values = GRID_VALUES
reliability_beta_text_values = GRID_VALUES
reliability_beta_visual_pos_values = GRID_VALUES
reliability_beta_visual_neg_values = GRID_VALUES

reliability_grid_results = []
for grid_alpha, grid_beta_text, grid_beta_visual_pos, grid_beta_visual_neg in itertools.product(
    reliability_alpha_values,
    reliability_beta_text_values,
    reliability_beta_visual_pos_values,
    reliability_beta_visual_neg_values,
):
    reliability_result = run_reliability_hybrid_retrieval(
        query_json=baseline_val_query_json,
        image_embedding_dir=baseline_img_embedding_val_dir,
        text_embedding_dir=hybrid_text_embedding_dir,
        visual_direction_dir=reliability_embedding_dir,
        alpha=grid_alpha,
        beta_text=grid_beta_text,
        beta_visual_pos=grid_beta_visual_pos,
        beta_visual_neg=grid_beta_visual_neg,
        batch_size=batch_size,
        device=device,
    )
    reliability_grid_results.append({
        "alpha": grid_alpha,
        "beta_text": grid_beta_text,
        "beta_visual_pos": grid_beta_visual_pos,
        "beta_visual_neg": grid_beta_visual_neg,
        **reliability_result["metrics"],
    })

reliability_grid_summary = pd.DataFrame(reliability_grid_results)
reliability_grid_summary.sort_values("recall@10", ascending=False).reset_index(drop=True)


,alpha,beta_text,beta_visual_pos,beta_visual_neg,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,0.50,1.00,0.25,0.25,0.035999,0.035999,0.116671,0.029022,0.178662,0.025286
1,1.00,2.00,0.50,0.50,0.035999,0.035999,0.116671,0.029022,0.178662,0.025286
2,1.00,2.00,0.50,0.25,0.035137,0.035137,0.117255,0.029166,0.177911,0.025213
3,1.50,2.00,1.00,1.00,0.035388,0.035388,0.114558,0.028788,0.176938,0.025489
4,0.50,1.50,0.25,0.50,0.035415,0.035415,0.114642,0.027932,0.176910,0.024552
...,...,...,...,...,...,...,...,...,...,...
620,0.25,1.00,2.00,2.00,0.005226,0.005226,0.022378,0.004970,0.040391,0.005090
621,0.25,0.50,2.00,1.50,0.006060,0.006060,0.022600,0.005187,0.039279,0.005029
622,0.25,0.50,2.00,2.00,0.006255,0.006255,0.022684,0.005215,0.039113,0.005020
623,0.25,0.25,2.00,1.50,0.006255,0.006255,0.022378,0.005171,0.039029,0.004926


In [ ]:
# The reliability-weighted gates selected on validation are evaluated once on test.
best_reliability = reliability_grid_summary.sort_values("recall@10", ascending=False).iloc[0]

reliability_test_result = run_reliability_hybrid_retrieval(
    query_json=baseline_test_query_json,
    image_embedding_dir=baseline_img_embedding_test_dir,
    text_embedding_dir=hybrid_text_embedding_dir,
    visual_direction_dir=reliability_embedding_dir,
    alpha=float(best_reliability["alpha"]),
    beta_text=float(best_reliability["beta_text"]),
    beta_visual_pos=float(best_reliability["beta_visual_pos"]),
    beta_visual_neg=float(best_reliability["beta_visual_neg"]),
    batch_size=batch_size,
    device=device,
)

pd.DataFrame([{
    "alpha": float(best_reliability["alpha"]),
    "beta_text": float(best_reliability["beta_text"]),
    "beta_visual_pos": float(best_reliability["beta_visual_pos"]),
    "beta_visual_neg": float(best_reliability["beta_visual_neg"]),
    **reliability_test_result["metrics"],
}])


,alpha,beta_text,beta_visual_pos,beta_visual_neg,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,0.5,1.0,0.25,0.25,0.034098,0.034098,0.109585,0.027302,0.168764,0.023826


### 3.5.1. Direction-decorrelation reliability-weighted hybrid fusion
This variant keeps the same reliability-weighted retrieval function, but replaces the visual-direction table with a decorrelated version. For each attribute direction, we find the most similar other visual directions and subtract their projection. The goal is to reduce directions that accidentally encode correlated CelebA attributes. Since the artifacts are cached, the projection step runs only when the decorrelated direction file is missing.


In [ ]:
# correlation-aware visual-direction helper functions
# This variant removes components aligned with highly similar visual directions.
def compute_direction_cosine_matrix(directions: torch.Tensor) -> torch.Tensor:
    """Compute pairwise cosine similarities between visual direction rows."""
    normalized = l2_normalize(directions.float())
    return normalized @ normalized.T


def decorrelate_direction(
    direction: torch.Tensor,
    confounder_directions: torch.Tensor,
    lambda_reg: float = 1e-4,
    epsilon: float = 1e-8,
) -> torch.Tensor:
    """Remove projection of one direction onto selected confounder directions."""
    if confounder_directions.numel() == 0:
        return l2_normalize(direction)

    c_matrix = l2_normalize(confounder_directions).T
    gram = c_matrix.T @ c_matrix
    regularized = gram + lambda_reg * torch.eye(gram.shape[0], dtype=gram.dtype)
    coefficients = torch.linalg.solve(regularized, c_matrix.T @ direction.float())
    cleaned = direction.float() - c_matrix @ coefficients
    return cleaned / (torch.linalg.norm(cleaned) + epsilon)


def compute_and_save_decorrelated_directions(
    visual_direction_dir: str | Path,
    output_dir: str | Path,
    visual_direction_name: str = "visual_directions_reliable",
    reliability_name: str = "visual_direction_reliability",
    output_direction_name: str = "decorrelated_visual_directions",
    direction_cosine_threshold: float = 0.70,
    max_confounders_per_attribute: int = 2,
) -> None:
    """Create conservative correlation-aware visual directions and cache them."""
    directions, direction_ids = load_embeddings(visual_direction_dir, visual_direction_name, "cpu")
    reliability, reliability_ids = load_embeddings(visual_direction_dir, reliability_name, "cpu")
    direction_ids = [str(attribute) for attribute in direction_ids]
    reliability_ids = [str(attribute) for attribute in reliability_ids]
    if direction_ids != reliability_ids:
        raise ValueError("Direction IDs and reliability IDs must have the same order.")

    directions = l2_normalize(directions.float())
    cosine_matrix = compute_direction_cosine_matrix(directions)
    cleaned_rows = []
    for row in range(len(direction_ids)):
        scores = cosine_matrix[row].clone()
        scores[row] = 0.0
        candidates = [
            index
            for index in torch.argsort(scores.abs(), descending=True).tolist()
            if abs(float(scores[index].item())) >= direction_cosine_threshold
        ][:max_confounders_per_attribute]
        cleaned_rows.append(
            decorrelate_direction(
                direction=directions[row],
                confounder_directions=directions[candidates] if candidates else directions.new_empty((0, directions.shape[1])),
            )
        )

    output_dir = Path(output_dir)
    save_embedding_files(torch.stack(cleaned_rows), direction_ids, output_dir, output_direction_name)
    save_embedding_files(reliability, reliability_ids, output_dir, reliability_name)


In [ ]:
# Decorrelated directions are cached after the first computation.
correlation_aware_dir = direction_decorrelation_dir

if not (correlation_aware_dir / "decorrelated_visual_directions.pt").exists():
    compute_and_save_decorrelated_directions(
        visual_direction_dir=reliability_embedding_dir,
        output_dir=correlation_aware_dir,
        direction_cosine_threshold=0.70,
        max_confounders_per_attribute=2,
    )

assert (correlation_aware_dir / "decorrelated_visual_directions.pt").exists()
assert (correlation_aware_dir / "visual_direction_reliability.pt").exists()


In [ ]:
# Correlation-aware reliability evaluation uses the same gates as the reliability model.
# The only change is the decorrelated visual-direction table.
reliability_alpha_values = GRID_VALUES
reliability_beta_text_values = GRID_VALUES
reliability_beta_visual_pos_values = GRID_VALUES
reliability_beta_visual_neg_values = GRID_VALUES

correlation_aware_grid_results = []
for grid_alpha, grid_beta_text, grid_beta_visual_pos, grid_beta_visual_neg in itertools.product(
    reliability_alpha_values,
    reliability_beta_text_values,
    reliability_beta_visual_pos_values,
    reliability_beta_visual_neg_values,
):
    correlation_aware_result = run_reliability_hybrid_retrieval(
        query_json=baseline_val_query_json,
        image_embedding_dir=baseline_img_embedding_val_dir,
        text_embedding_dir=hybrid_text_embedding_dir,
        visual_direction_dir=correlation_aware_dir,
        visual_direction_name="decorrelated_visual_directions",
        alpha=grid_alpha,
        beta_text=grid_beta_text,
        beta_visual_pos=grid_beta_visual_pos,
        beta_visual_neg=grid_beta_visual_neg,
        batch_size=batch_size,
        device=device,
    )
    correlation_aware_grid_results.append({
        "alpha": grid_alpha,
        "beta_text": grid_beta_text,
        "beta_visual_pos": grid_beta_visual_pos,
        "beta_visual_neg": grid_beta_visual_neg,
        "direction_cosine_threshold": 0.70,
        "max_confounders": 2,
        **correlation_aware_result["metrics"],
    })

correlation_aware_grid_summary = pd.DataFrame(correlation_aware_grid_results)
correlation_aware_grid_summary.sort_values("recall@10", ascending=False).reset_index(drop=True)


,alpha,beta_text,beta_visual_pos,beta_visual_neg,direction_cosine_threshold,max_confounders,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,1.50,1.00,2.0,2.00,0.7,2,0.038279,0.038279,0.126623,0.031802,0.196731,0.028266
1,1.00,1.50,1.0,1.00,0.7,2,0.038223,0.038223,0.127429,0.031768,0.196147,0.027968
2,1.50,0.50,2.0,2.00,0.7,2,0.037917,0.037917,0.127346,0.031907,0.195536,0.028077
3,1.50,1.50,2.0,1.50,0.7,2,0.038584,0.038584,0.127846,0.032124,0.195119,0.028193
4,1.50,2.00,1.5,1.50,0.7,2,0.038223,0.038223,0.127623,0.031891,0.195007,0.027871
...,...,...,...,...,...,...,...,...,...,...,...,...
620,0.25,0.25,2.0,2.00,0.7,2,0.015539,0.015539,0.061518,0.014005,0.093765,0.011940
621,0.25,2.00,1.5,0.50,0.7,2,0.015706,0.015706,0.059600,0.014311,0.091791,0.012318
622,0.25,2.00,2.0,0.25,0.7,2,0.014928,0.014928,0.060128,0.014333,0.090012,0.012254
623,0.25,2.00,2.0,1.00,0.7,2,0.015790,0.015790,0.061157,0.014266,0.089984,0.011809


In [ ]:
# The correlation-aware setting is evaluated once on the held-out test split.
best_correlation_aware = correlation_aware_grid_summary.sort_values("recall@10", ascending=False).iloc[0]

correlation_aware_test_result = run_reliability_hybrid_retrieval(
    query_json=baseline_test_query_json,
    image_embedding_dir=baseline_img_embedding_test_dir,
    text_embedding_dir=hybrid_text_embedding_dir,
    visual_direction_dir=correlation_aware_dir,
    visual_direction_name="decorrelated_visual_directions",
    alpha=float(best_correlation_aware["alpha"]),
    beta_text=float(best_correlation_aware["beta_text"]),
    beta_visual_pos=float(best_correlation_aware["beta_visual_pos"]),
    beta_visual_neg=float(best_correlation_aware["beta_visual_neg"]),
    batch_size=batch_size,
    device=device,
)

pd.DataFrame([{
    "alpha": float(best_correlation_aware["alpha"]),
    "beta_text": float(best_correlation_aware["beta_text"]),
    "beta_visual_pos": float(best_correlation_aware["beta_visual_pos"]),
    "beta_visual_neg": float(best_correlation_aware["beta_visual_neg"]),
    "direction_cosine_threshold": float(best_correlation_aware["direction_cosine_threshold"]),
    "max_confounders": int(best_correlation_aware["max_confounders"]),
    **correlation_aware_test_result["metrics"],
}])


,alpha,beta_text,beta_visual_pos,beta_visual_neg,direction_cosine_threshold,max_confounders,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,1.5,1.0,2.0,2.0,0.7,2,0.033553,0.033553,0.107679,0.02706,0.169037,0.023932


In [ ]:
# Build the results summary table from the final test result objects.
def summary_row(experiment: str, result: dict[str, Any]) -> dict[str, float | str]:
    metrics = result["metrics"]
    return {
        "Experiment": experiment,
        "Recall@1": metrics["recall@1"],
        "Precision@1": metrics["precision@1"],
        "Recall@5": metrics["recall@5"],
        "Precision@5": metrics["precision@5"],
        "Recall@10": metrics["recall@10"],
        "Precision@10": metrics["precision@10"],
    }

results_summary = pd.DataFrame([
    summary_row("CLIP Arithmetic Baseline", baseline_result),
    summary_row("Gated Text Fusion", gs_test_result),
    summary_row("Visual Direction Fusion", visual_test_result),
    summary_row("Hybrid Text + Visual Fusion", hybrid_test_result),
    summary_row("Prompt Averaging Hybrid", prompt_average_test_result),
    summary_row("Prompt Difference Hybrid", prompt_difference_test_result),
    summary_row("Reliability-Weighted Hybrid", reliability_test_result),
    summary_row("Direction-Decorrelation Reliability Hybrid", correlation_aware_test_result),
])

metric_columns = [column for column in results_summary.columns if column != "Experiment"]
results_summary_display = results_summary.copy()
results_summary_display[metric_columns] = results_summary_display[metric_columns].round(3)
results_summary_display


## Results Summary

The table above is generated from the final test result objects produced by each experiment section. Values are rounded to three decimals for display.


## Saturation analysis

| Step | Recall@10 | Gain over previous | Gain over baseline |
|---|---:|---:|---:|
| CLIP Arithmetic Baseline | 0.116 | - | - |
| Gated Text Fusion | 0.138 | +0.022 | +0.022 |
| Visual Direction Fusion | 0.147 | +0.009 | +0.031 |
| Hybrid Text + Visual Fusion | 0.164 | +0.017 | +0.048 |
| Prompt Difference Hybrid | 0.174 | +0.010 | +0.058 |
| Reliability-Weighted Hybrid | 0.169 | -0.005 | +0.053 |
| Direction-Decorrelation Reliability Hybrid | 0.169 | +0.000 | +0.053 |

The largest gains occur when moving from naive CLIP arithmetic to gated and hybrid fusion. Later fixed-weight variants produce only marginal changes, suggesting that the main bottleneck is the rigidity of fixed global fusion rules rather than the absence of another handcrafted weighting variant.

# 4. Discussion and conclusion

# 4. Discussion and conclusion

Overall, the experiments show that progressively more structured fusion mechanisms improve over naive CLIP arithmetic, but the gains saturate once fixed-weight hybrid fusion is introduced.

The CLIP arithmetic baseline reaches Recall@10 = 0.116. Gated text fusion improves this to 0.138, showing that positive and negative conditions should not necessarily be weighted equally. Replacing text conditions with CelebA visual directions further improves performance to 0.147, suggesting that dataset-specific visual directions contain useful attribute information.

The strongest improvement comes from combining CLIP text semantics with CelebA visual directions. The original hybrid text-and-visual fusion reaches Recall@10 = 0.164, while the best prompt-difference hybrid reaches Recall@10 = 0.174. This corresponds to an approximate 49.6% relative improvement over the CLIP arithmetic baseline.

However, the later variants reveal a saturation behavior. Prompt averaging, reliability weighting, and direction decorrelation only produce small additional gains or fail to improve over prompt differencing. This suggests that the main limitation is not simply the choice of fixed scalar weights, but the rigidity of using one global fusion rule across all query types and attributes. Some attributes, such as Eyeglasses or Mustache, may benefit from stronger visual directions, while more ambiguous or correlated attributes, such as Young, Chubby, or Heavy Makeup, likely require query-adaptive reasoning.

Overall, the project demonstrates that training-free hybrid text-visual conditioning can substantially improve over naive CLIP arithmetic, but also identifies the limitation of fixed global fusion mechanisms. Larger improvements would likely require learned or query-adaptive fusion modules with stronger regularization and a more systematic validation protocol.

# 5. References

Liang, V. W., Zhang, Y., Kwon, Y., Yeung, S., & Zou, J. Y. (2022). Mind the gap: Understanding the modality gap in multi-modal contrastive representation learning. Advances in Neural Information Processing Systems, 35, 17612-17625.

Radford, A., Kim, J. W., Hallacy, C., Ramesh, A., Goh, G., Agarwal, S., ... & Sutskever, I. (2021, July). Learning transferable visual models from natural language supervision. In International conference on machine learning (pp. 8748-8763). PmLR.

Patashnik, O., Wu, Z., Shechtman, E., Cohen-Or, D., & Lischinski, D. (2021). Styleclip: Text-driven manipulation of stylegan imagery. In Proceedings of the IEEE/CVF international conference on computer vision (pp. 2085-2094).

Wei, T., Chen, D., Zhou, W., Liao, J., Tan, Z., Yuan, L., Zhang, W., & Yu, N. (2022). HairCLIP: Design your hair by text and reference image. In Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition (pp. 18072-18081).

Bhalla, U., Oesterling, A., Srinivas, S., Calmon, F. P., & Lakkaraju, H. (2024). Interpreting CLIP with sparse linear concept embeddings. arXiv preprint arXiv:2402.10376.

Yang, Y., Nushi, B., Palangi, H., & Mirzasoleiman, B. (2023). Mitigating spurious correlations in multi-modal models during fine-tuning. arXiv preprint arXiv:2304.03916.

Zhao, J., Li, C., Sala, F., & Rohe, K. (2025). Quantifying structure in CLIP embeddings: A statistical framework for concept interpretation. arXiv preprint arXiv:2506.13831.